In [ ]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 726.2 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 63.0 MB/s eta 0:00:00


In [ ]:
# ==========================================================
# PHASE 1A
# AUTOMATIC PARAMETER DISCOVERY FROM TOC
# ==========================================================

import pdfplumber
import pandas as pd
import re
import json

PDF_PATH = "/content/co-ursi-key-regultry-parmtrs-pwr-utiltis-dt180526.pdf"

# ----------------------------------------------------------
# Extract first 15 pages
# TOC is usually there
# ----------------------------------------------------------

toc_text = []

with pdfplumber.open(PDF_PATH) as pdf:

    max_pages = min(15, len(pdf.pages))

    for i in range(max_pages):

        text = pdf.pages[i].extract_text()

        if text:
            toc_text.append(text)

toc_text = "\n".join(toc_text)

print("TOC text length:", len(toc_text))

# ----------------------------------------------------------
# Find Table Entries
# Example:
#
# TABLE-5(E) : BANKING CHARGES ..... 63
#
# ----------------------------------------------------------

pattern = re.compile(
    r"TABLE[- ]?\d+(?:\([A-Z]\))?\s*:\s*(.*?)\s+(\d+)",
    re.IGNORECASE
)

matches = pattern.findall(toc_text)

print("\nTotal TOC matches found:", len(matches))

# ----------------------------------------------------------
# Build Parameter Catalog
# ----------------------------------------------------------

catalog = {}

for title, page in matches:

    title = " ".join(title.split())

    catalog[title] = {
        "start_page": int(page)
    }

# ----------------------------------------------------------
# Save
# ----------------------------------------------------------

with open("parameter_catalog.json", "w") as f:
    json.dump(catalog, f, indent=4)

# ----------------------------------------------------------
# Display
# ----------------------------------------------------------

df = pd.DataFrame([
    {
        "parameter": k,
        "start_page": v["start_page"]
    }
    for k, v in catalog.items()
])

df = df.sort_values("start_page")

print("\n===== DISCOVERED PARAMETERS =====")
display(df)

print("\nSaved:")
print("parameter_catalog.json")

TOC text length: 59069

Total TOC matches found: 30

===== DISCOVERED PARAMETERS =====


,parameter,start_page
0,TARIFF ORDER FOR FY-27 AND TRUE UP ORDER FOR F...,22
1,ISSUANCE OF MULTI YEAR TARIFF (MYT) ORDER .......,29
2,CATEGORY WISE COMPARISON OF REPRESENTATIVE TAR...,31
4,NON-DOMESTIC ....................................,35
5,AGRICULTURE .....................................,37
6,SMALL INDUSTRIAL ................................,39
7,MEDIUM INDUSTRIAL ...............................,41
8,LARGE INDUSTRIAL ................................,43
9,CROSS SUBSIDY - LT INDUSTRIAL CONSUMERS .. ......,45
10,CROSS SUBSIDY - HT INDUSTRIAL CONSUMERS .........,47



Saved:
parameter_catalog.json


In [ ]:
# ==========================================================
# PHASE 1B
# PARAMETER RANGE GENERATOR
# ==========================================================

import json
import pandas as pd

# ----------------------------------------------------------
# Load catalog
# ----------------------------------------------------------

with open("parameter_catalog.json", "r") as f:
    catalog = json.load(f)

# ----------------------------------------------------------
# Sort by start page
# ----------------------------------------------------------

items = sorted(
    catalog.items(),
    key=lambda x: x[1]["start_page"]
)

# ----------------------------------------------------------
# Create page ranges
# ----------------------------------------------------------

parameter_ranges = {}

for i, (name, info) in enumerate(items):

    start_page = info["start_page"]

    if i < len(items) - 1:
        next_start = items[i+1][1]["start_page"]
        end_page = next_start - 1
    else:
        end_page = None

    parameter_ranges[name] = {
        "start_page": start_page,
        "end_page": end_page
    }

# ----------------------------------------------------------
# Save
# ----------------------------------------------------------

with open("parameter_ranges.json", "w") as f:
    json.dump(parameter_ranges, f, indent=4)

# ----------------------------------------------------------
# Display
# ----------------------------------------------------------

rows = []

for name, info in parameter_ranges.items():

    rows.append({
        "parameter": name,
        "start_page": info["start_page"],
        "end_page": info["end_page"]
    })

df = pd.DataFrame(rows)

print("Total parameters:", len(df))
display(df)

print("\nSaved: parameter_ranges.json")

Total parameters: 29


,parameter,start_page,end_page
0,TARIFF ORDER FOR FY-27 AND TRUE UP ORDER FOR F...,22,28.0
1,ISSUANCE OF MULTI YEAR TARIFF (MYT) ORDER .......,29,30.0
2,CATEGORY WISE COMPARISON OF REPRESENTATIVE TAR...,31,34.0
3,NON-DOMESTIC ....................................,35,36.0
4,AGRICULTURE .....................................,37,38.0
5,SMALL INDUSTRIAL ................................,39,40.0
6,MEDIUM INDUSTRIAL ...............................,41,42.0
7,LARGE INDUSTRIAL ................................,43,44.0
8,CROSS SUBSIDY - LT INDUSTRIAL CONSUMERS .. ......,45,46.0
9,CROSS SUBSIDY - HT INDUSTRIAL CONSUMERS .........,47,48.0



Saved: parameter_ranges.json


In [ ]:
# ==========================================================
# PHASE 1C
# VERIFY PARAMETER START PAGES
# ==========================================================

import pdfplumber
import pandas as pd

PDF_PATH = "/content/co-ursi-key-regultry-parmtrs-pwr-utiltis-dt180526.pdf"

TARGET_PAGES = [
    49,
    56,
    58,
    61,
    63,
    75,
    80,
    88,
    105,
    117,
    119
]

results = []

with pdfplumber.open(PDF_PATH) as pdf:

    for page_num in TARGET_PAGES:

        page = pdf.pages[page_num - 1]

        text = page.extract_text() or ""

        first_lines = "\n".join(
            text.split("\n")[:15]
        )

        results.append({
            "page": page_num,
            "preview": first_lines
        })

df = pd.DataFrame(results)

pd.set_option("display.max_colwidth", 500)

display(df)

,page,preview
0,49,"रा(cid:475)/क(cid:336)(cid:363) शािसत (cid:366)देश (cid:354)ॉस-सबिसडी / Cross-Subsidy (in %) ऊजा(cid:330) की खपत / Energy Consumption (in MUs)\nState/ UT 2023-24 2024-25 2025-26 2026-27 2023-24 2024-25 2025-26 2026-27\nमिणपुर / Manipur 20% 19% 44% N/A 15 30 18 N/A\nमेघालय / Meghalaya 0% -5% -1% 1% 679 1,106 1,106 631\nिमजोरम / Mizoram 19% 5% 8% 19% 6 10 16 7\nनागाल(cid:339)ड / Nagaland NA NA 11% N/A 46 10 10 N/A\nओिडशा / Odisha N/A N/A N/A N/A N/A N/A N/A N/A\nपुडुचेरी / Puducherry 14% 13% 8..."
1,56,Industries (Urban) 1.60\nPrivate School 1.62\nDomestic (50 KVA & above) 1.41\nIndustries (50 KVA & above) 1.69\nCommercial (50 KVA & above) 1.71\nHV & EHV\nDomestic 1.09\nCommercial-11kV and 33kV 1.71\nIndustries-11kV 1.55\nIndustries-33kV 1.46\nIndustries-132kV 1.00\nPublic Utility & Water Works 1.70\nCold Storage 0.63\nMetro 0.75\nMES 0.49
2,58,रा(cid:475)/के(cid:574) शािसत (cid:366)देश लागू अविध अित(cid:303)र(cid:389) अिधभार\nStates/UTs Applicable Period Additional Surcharge\nम(cid:559)म अित(cid:303)र(cid:389) अिधभार (cid:721)र: > 1.50 (cid:348)पये से 2.00 (cid:348)पये तक\nMedium Additional Surcharge Level: > Rs 1.50 to Rs. 2.00\n(Oct-Apr): 1.33-1.90\nिद(cid:821) ली / Delhi 2021-22\n(May-Sep): 0.66-0.95\n(Oct-Apr): 1.70\nबीआरपीएल / BRPL 2021-22\n(May-Sep): 0.85\n(Oct-Apr): 1.33\nबीवाईपीएल / BYPL 2021-22\n(May-Sep): 0.66\n(Oct-Apr)...
3,61,(cid:684)ीिलंग शु(cid:651) / Wheeling Charges\nवो(cid:656)ेज (cid:721)र / Voltage Level\nलागू\nअविध 200 केवी\n11 केवी से\nरा(cid:475)/क(cid:336)(cid:363) शािसत (cid:366)देश / States/UTs (िव(cid:517) वष(cid:330)) नीचे 11 केवी 33 केवी 66 केवी 132 केवी और उससे\nApplicable अिधक\nBelow 11 kV 33 kV 66 kV 132 kV\nPeriod 200 kV &\n11 kV\n(FY) Above\nमेघालय / Meghalaya 2026-27 2.68\nएमईपीडीसीएल / MePDCL 2026-27 2.68\nिमजोरम / Mizoram 2026-27 1.24\nिमजोरम पीडी / Mizoram PD 2026-27 1.24
4,63,"दीघ(cid:330)कािलक एवं\nलागू वष(cid:330) म(cid:559)म अविध अ(cid:665)कािलक\nरा(cid:475)/क(cid:336)(cid:363) शािसत (cid:366)देश / इकाई इकाई\nApplicable Long & Short\nStates/UTs Unit Unit\nYear Medium Term\nTerm\nकेरल / Kerala 2024-25 0.49 Rs/kWh 10,925.00 Rs/MW/Day\nकेएसईबीएल / KSEBL 2024-25 0.49 Rs/kWh 10,565.00 Rs/MW/Day\nम(cid:559) (cid:366)देश / Madhya Pradesh 2023-24 4,686.91 Rs/MW/Day 120.63 Rs/MWh\nएमपीपीटीसीएल / MPPTCL 2023-24 4,686.91 Rs/MW/Day 120.63 Rs/MWh\nमहारा(cid:700)(cid:332)* /..."
5,75,"ब(cid:339)िकंग\nब(cid:339)िकंग शु(cid:651)ो ंका िववरण\nरा(cid:475) िवतरण कंपनी\nअविध\nअ(cid:366)यु(cid:389) संिचत ऊजा(cid:330) का िनपटान\nDescription of\nState DISCOM Banking Treatment of unutilized banked energy\nBanking Charges\nPeriod\n(cid:366)(cid:529)ेक िबिलंग आवत(cid:330)न के अंत म(cid:336) उपयोग न की गई अिधशेष ऊजा(cid:330) को समा(cid:593) माना जाएगा। हालांिक,\nउ(cid:517)राखंड Monthly/ नवीकरणीय ऊजा(cid:330) उ(cid:523)ादन क(cid:336)(cid:363) उस सीमा तक नवीकरणीय ऊजा(cid:330) (cid:366)मा..."
6,80,"संबंिधत उपभो(cid:389)ा (cid:373)ेणी म(cid:336) टीओडी की (cid:366)यो(cid:475)ता\nवष(cid:330) TOD applicability across respective consumer category\nरा(cid:475) Year\nएचटी और ईएचटी एचटी और ईएचटी\nState (िव(cid:517) वष(cid:330))\nघरेलू एलटी कमिश(cid:330)यल एलटी औ(cid:552)ोिगक एलटी अ(cid:586) कृिष एचटी कमिश(cid:330)यल औ(cid:552)ोिगक अ(cid:586)\n(FY)\nDomestic LT Commercial LT Industrial LT Others Agriculture HT Commercial HT & EHT HT & EHT\nIndustrial Others\nYes\nYes\nEmergency Supply,\nYes\nCo..."
7,88,अनुमोिदत\nकर प(cid:686)ात आरओई की टै(cid:303)रफ िविनयमो ंके अनुसार\nवष(cid:330) आरओई\nरा(cid:475)/के(cid:574) शािसत (cid:366)देश अनुमोिदत दर कर प(cid:686)ात (cid:623)ाज दर\nYear (करोड़ (cid:348)पये म(cid:336))\nStates/UTs Approved rate of Rate of post-tax RoE\n(िव(cid:517) वष(cid:330)) Approved RoE\npost-tax RoE as per tariff regulations\n(FY) (in Rs. Cr)\nजेवीवीएनएल / JVVNL 2026-27 0.00 0.00% 16.00%\nिस(cid:304)(cid:383)म / Sikkim 2025-26 0 0.00% 14.00%\nिस(cid:304)(cid:383)म पीडी / Sikkim P...
8,105,िव(cid:517) अ(cid:738)ीकृितयां\nFinancial Disallowances\nनेट एआरआर नेट एआरआर के\nनेट एआरआर\n((cid:366)(cid:

In [ ]:
# ==========================================================
# FIND PAGE NUMBER OFFSET
# ==========================================================

import pdfplumber
import re

PDF_PATH = "/content/co-ursi-key-regultry-parmtrs-pwr-utiltis-dt180526.pdf"

targets = [
    "CROSS SUBSIDY SURCHARGE",
    "ADDITIONAL SURCHARGE",
    "WHEELING CHARGE",
    "TRANSMISSION CHARGE",
    "BANKING CHARGES",
    "GREEN ENERGY OPEN ACCESS",
    "RELIABILITY OF SUPPLY"
]

with pdfplumber.open(PDF_PATH) as pdf:

    for pdf_idx, page in enumerate(pdf.pages):

        text = page.extract_text()

        if not text:
            continue

        text_upper = text.upper()

        for target in targets:

            if target in text_upper:

                print("\n" + "="*80)
                print("FOUND:", target)
                print("PDF PAGE:", pdf_idx + 1)

                first_lines = "\n".join(
                    text.split("\n")[:20]
                )

                print(first_lines[:1000])
                print("="*80)


FOUND: CROSS SUBSIDY SURCHARGE
PDF PAGE: 5
Table of Contents
1. BACKGROUND ....................................................................................................... 12
2. LIST OF PARAMETERS FOR COMPLIANCE STATUS OF REGULATORY PARAMETERS .. 13
3. KEY UPDATES OF THE QUARTER……………………………………………...…………19
4. COMPLIANCE STATUS OF REGULATORY PARAMETERS ACROSS STATES ................. 21
TABLE-1(A) : TARIFF ORDER FOR FY-27 AND TRUE UP ORDER FOR FY-25 FOR DISTRIBUTION UTILITIES .................... 22
TABLE-1(B) : ISSUANCE OF MULTI YEAR TARIFF (MYT) ORDER ................................................................................. 29
TABLE-2 : CATEGORY WISE COMPARISON OF REPRESENTATIVE TARIFFS .............................................................. 31
TABLE-2(A) : DOMESTIC-3KW 100 UNITS .................................................................................................................. 31
TABLE-2(B) : DOMESTIC-3KW 500 UNITS .................................

In [ ]:
# ==========================================================
# PHASE 2A
# UNIVERSAL PARAMETER FAMILY EXTRACTOR
# ==========================================================

import pdfplumber
import json
import pandas as pd

PDF_PATH = "/content/co-ursi-key-regultry-parmtrs-pwr-utiltis-dt180526.pdf"

# ----------------------------------------------------------
# Banking Charges
# ----------------------------------------------------------

START_PAGE = 64
END_PAGE = 75

all_tables = []
summary = []

with pdfplumber.open(PDF_PATH) as pdf:

    for page_num in range(START_PAGE, END_PAGE + 1):

        page = pdf.pages[page_num - 1]

        tables = page.extract_tables()

        summary.append({
            "page": page_num,
            "tables_found": len(tables)
        })

        for table_idx, table in enumerate(tables):

            all_tables.append({
                "page": page_num,
                "table_id": table_idx,
                "rows": table
            })

# ----------------------------------------------------------
# Save
# ----------------------------------------------------------

with open("banking_charge_raw.json", "w") as f:
    json.dump(all_tables, f, indent=2, ensure_ascii=False)

summary_df = pd.DataFrame(summary)

print("="*80)
print("TOTAL PAGES:", END_PAGE - START_PAGE + 1)
print("TOTAL TABLES:", len(all_tables))
print("="*80)

display(summary_df)

# ----------------------------------------------------------
# Preview first table
# ----------------------------------------------------------

if len(all_tables) > 0:

    first_table = all_tables[0]["rows"]

    print("\nFIRST TABLE PREVIEW:\n")

    for row in first_table[:10]:
        print(row)

else:
    print("NO TABLES DETECTED")

TOTAL PAGES: 12
TOTAL TABLES: 60


,page,tables_found
0,64,5
1,65,5
2,66,5
3,67,5
4,68,5
5,69,5
6,70,5
7,71,5
8,72,5
9,73,5



FIRST TABLE PREVIEW:

['रा(cid:475)\nState', 'िवतरण कंपनी\nDISCOM', 'ब(cid:339)िकंग शु(cid:651)ो ंका िववरण\nDescription of\nBanking Charges', '', 'ब(cid:339)िकंग', '', 'अ(cid:366)यु(cid:389) संिचत ऊजा(cid:330) का िनपटान\nTreatment of unutilized banked energy']
[None, None, None, None, 'अविध', None, None]
[None, None, None, None, 'Banking', None, None]
[None, None, None, None, 'Period', None, None]
['अंडमान और\nिनकोबार (cid:554)ीप\nसमूह\nAndaman &\nNicobar Islands', 'ED-A&NI', '8.00%', 'Monthly/\nBilling\nCycle', None, None, "1. नवीकरणीय ऊजा(cid:330) उ(cid:523)ादन क(cid:336)(cid:363) (cid:554)ारा िकए गए कुल उ(cid:523)ादन के 20% तक सीिमत: शेष बची िवद्युत,\nकी 'डी(cid:817)ड खरीद' मानी जाएगी, िजसका भुगतान औसत िवद्यु(cid:514)रीद लागत (एपीपीसी) या उस वष(cid:330) के िलए\nिनधा(cid:330)(cid:303)रत फीड-इन-टै(cid:303)रफ (िबना स(cid:304)(cid:629)डी और (cid:533)(cid:303)रत मू(cid:670)(cid:376)ास के लाभ को जोड़े), दोनो ंम(cid:336) से जो भी कम\nहो, उस दर पर िकया जाएगा।\nLimited to 30% of the total ge

In [ ]:
# ==========================================================
# PHASE 2B
# INSPECT TABLE SHAPES
# ==========================================================

import pdfplumber
import pandas as pd

PDF_PATH = "/content/co-ursi-key-regultry-parmtrs-pwr-utiltis-dt180526.pdf"

START_PAGE = 64
END_PAGE = 75

rows = []

with pdfplumber.open(PDF_PATH) as pdf:

    for page_num in range(START_PAGE, END_PAGE + 1):

        page = pdf.pages[page_num - 1]

        tables = page.extract_tables()

        for table_idx, table in enumerate(tables):

            n_rows = len(table)

            n_cols = max(
                len(r)
                for r in table
                if r
            )

            sample = ""

            for row in table[:2]:

                sample += str(row)[:150]

            rows.append({
                "page": page_num,
                "table_id": table_idx,
                "rows": n_rows,
                "cols": n_cols,
                "sample": sample
            })

df = pd.DataFrame(rows)

pd.set_option("display.max_colwidth", 200)

display(df)

,page,table_id,rows,cols,sample
0,64,0,8,7,"['रा(cid:475)\nState', 'िवतरण कंपनी\nDISCOM', 'ब(cid:339)िकंग शु(cid:651)ो ंका िववरण\nDescription of\nBanking Charges', '', 'ब(cid:339)िकंग', '', 'अ(c[None, None, None, None, 'अविध', None, None]"
1,64,1,3,1,['ब(cid:339)िकंग शु(cid:651)ो ंका िववरण']['Description of']
2,64,2,2,1,['रा(cid:475)']['State']
3,64,3,2,1,['िवतरण कंपनी']['DISCOM']
4,64,4,2,1,['अ(cid:366)यु(cid:389) संिचत ऊजा(cid:330) का िनपटान']['Treatment of unutilized banked energy']
5,65,0,10,7,"['रा(cid:475)\nState', 'िवतरण कंपनी\nDISCOM', 'ब(cid:339)िकंग शु(cid:651)ो ंका िववरण\nDescription of\nBanking Charges', '', 'ब(cid:339)िकंग', '', 'अ(c[None, None, None, None, 'अविध', None, None]"
6,65,1,3,1,['ब(cid:339)िकंग शु(cid:651)ो ंका िववरण']['Description of']
7,65,2,2,1,['रा(cid:475)']['State']
8,65,3,2,1,['िवतरण कंपनी']['DISCOM']
9,65,4,2,1,['अ(cid:366)यु(cid:389) संिचत ऊजा(cid:330) का िनपटान']['Treatment of unutilized banked energy']


In [ ]:
# ==========================================================
# PHASE 2C
# MERGE MULTI-PAGE BANKING TABLE
# ==========================================================

import pdfplumber
import pandas as pd

PDF_PATH = "/content/co-ursi-key-regultry-parmtrs-pwr-utiltis-dt180526.pdf"

START_PAGE = 64
END_PAGE = 75

merged_rows = []

header = None

with pdfplumber.open(PDF_PATH) as pdf:

    for page_num in range(START_PAGE, END_PAGE + 1):

        page = pdf.pages[page_num - 1]

        tables = page.extract_tables()

        if not tables:
            continue

        # --------------------------------------------------
        # keep largest table only
        # --------------------------------------------------

        largest_table = max(
            tables,
            key=lambda t: len(t) * max(len(r) for r in t if r)
        )

        if len(largest_table) < 2:
            continue

        # --------------------------------------------------
        # first page header
        # --------------------------------------------------

        if header is None:
            header = largest_table[:4]

        # --------------------------------------------------
        # skip repeated header rows
        # --------------------------------------------------

        data_rows = largest_table[4:]

        merged_rows.extend(data_rows)

print("=" * 80)
print("TOTAL DATA ROWS:", len(merged_rows))
print("=" * 80)

df = pd.DataFrame(merged_rows)

print("\nShape:")
print(df.shape)

print("\nFirst 15 rows:")
display(df.head(15))

print("\nLast 15 rows:")
display(df.tail(15))

# save inspection file
df.to_csv("banking_charge_merged_preview.csv", index=False)

print("\nSaved: banking_charge_merged_preview.csv")

TOTAL DATA ROWS: 63

Shape:
(63, 7)

First 15 rows:


,0,1,2,3,4,5,6
0,अंडमान और\nिनकोबार (cid:554)ीप\nसमूह\nAndaman &\nNicobar Islands,ED-A&NI,8.00%,Monthly/\nBilling\nCycle,None,None,"1. नवीकरणीय ऊजा(cid:330) उ(cid:523)ादन क(cid:336)(cid:363) (cid:554)ारा िकए गए कुल उ(cid:523)ादन के 20% तक सीिमत: शेष बची िवद्युत,\nकी 'डी(cid:817)ड खरीद' मानी जाएगी, िजसका भुगतान औसत िवद्यु(cid:5..."
1,आं(cid:364) (cid:366)देश\nAndhra\nPradesh,APEPDCL,8.00%,Monthly/\nBilling\nCycle,None,None,(cid:356)ीन एनज(cid:334) ओपन ए(cid:411)ेस के उपभो(cid:389)ा िबिलंग अविध के दौरान िवतरण लाइस(cid:336)सधारी से अपनी कुल मािसक\nिबजली खपत का केवल तीस (cid:366)ितशत (30%) ही जमा कर सक(cid:336)गे। 30% ...
2,None,APSPDCL,None,None,None,None,None
3,None,APCPDCL,None,None,None,None,None
4,,,,,None,None,"Provided further that, the lapsed unutilized surplus banked energy enlisted for RECs, if not\nclaimed by the renewable energy generating station, the DISCOMs shall account for such\nlapsed energy ..."
5,अ(cid:348)णाचल (cid:366)देश\nArunachal\nPradesh,DoP,8.00%,Monthly/\nBilling\nCycle,None,None,"(cid:356)ीन एनज(cid:334) ओपन ए(cid:411)ेस उपभो(cid:389)ाओ ंके िलए ब(cid:339)िकंग की गई ऊजा(cid:330) की अनुमत मा(cid:361)ा , उपभो(cid:389)ा (cid:554)ारा िवतरण\nलाइस(cid:336)सधारी से की गई कुल मािसक..."
6,असम\nAssam,APDCL,8.00%,Billing\nCycle,None,None,"(cid:366)(cid:529)ेक िबिलंग आवत(cid:330)न के अंत म(cid:336), समा(cid:593) माना जाएगा और नवीकरणीय ऊजा(cid:330) उ(cid:523)ादन क(cid:336)(cid:363) उस सीमा तक\nनवीकरणीय ऊजा(cid:330) (cid:366)माणप(cid:..."
7,िबहार\nBihar,NBPDCL,"8%\nDuring Apr to Oct, drawl\nof banked energy during\npeak TOD period shall\nnot be permitted.",Monthly,None,None,"(cid:366)(cid:529)ेक िबिलंग आवत(cid:330)न के अंत म(cid:336), समा(cid:593) माना जाएगा और नवीकरणीय ऊजा(cid:330) उ(cid:523)ादन क(cid:336)(cid:363) उस सीमा तक\nनवीकरणीय ऊजा(cid:330) (cid:366)माणप(cid:..."
8,None,SBPDCL,None,None,None,None,None
9,चंडीगढ़\nChandigarh,CPDL,8.00%,Monthly/\nBilling\nCycle,None,None,"1. नवीकरणीय ऊजा(cid:330) उ(cid:523)ादन क(cid:336)(cid:363) (cid:554)ारा िकए गए कुल उ(cid:523)ादन के 20% तक सीिमत: शेष बची िवद्युत, की\n'डी(cid:817)ड खरीद' मानी जाएगी, िजसका भुगतान औसत िवद्युत् खरी..."



Last 15 rows:


,0,1,2,3,4,5,6
48,राज(cid:725)ान\nRajasthan,AVVNL,8.00%,Monthly/\nBilling\nCycle,None,None,"िबिलंग आवत(cid:330)न की समा(cid:304)(cid:593) पर उपयोग न की गई ब(cid:339)क की गई ऊजा(cid:330) समा(cid:593) हो जाएगी और नवीकरणीय ऊजा(cid:330)\nकैि(cid:592)व उ(cid:523)ादन क(cid:336)(cid:363), िवद्य..."
49,,JVVNL,,,None,None,Unutilized banked energy at the end of billing cycle shall lapse and the renewable energy\ncaptive generating plant shall be entitled to get Renewable Energy Certificates to the extent\nof the lap...
50,None,JdVVNL,None,None,None,None,None
51,िस(cid:304)(cid:383)म\nSikkim,Power Department,8.00%,Monthly,None,None,"महीने के अंत म(cid:336) उपयोग न की गई अिधशेष ब(cid:339)क की गई ऊजा(cid:330) को (cid:366)(cid:529)ेक ब(cid:339)िकंग आवत(cid:330)न के अंत म(cid:336) समा(cid:593)\nमाना जाएगा; बशत(cid:337) िक, नवीकरण..."
52,तिमलनाडु\nTamil Nadu,TNPDCL,8.00%,Monthly,None,None,ब(cid:339)िकंग अविध के अंत म(cid:336) अ(cid:366)यु(cid:389) अिधशेष ब(cid:339)(cid:779)ड ऊजा(cid:330) को िवतरण लाइस(cid:336)सधारी को आयोग के आदेशो ंके\nअनुसार लागू संबंिधत आरई दरो ंके 75% की दर पर ...
53,,,,,None,None,"(ii) the preferential tariff determined by the Commission for the respective control period\nduring which the RE Generation Projects were commissioned,\nwhichever is lower.\nयिद संबंिधत वष(cid:330..."
54,तेलंगाना\nTelangana,TGSPDCL,8.00%,Monthly,None,None,"अ(cid:366)यु(cid:389) ऊजा(cid:330) (cid:366)(cid:529)ेक कैल(cid:336)डर माह के अंत म(cid:336) (cid:681)पगत मानी जाएगी: बशत(cid:337) िक, िजयोए उपभो(cid:389)ा उस सीमा\nतक नवीकरणीय ऊजा(cid:330) (cid:3..."
55,None,TGNPDCL,None,None,None,None,None
56,,,,,None,None,"in clause 33.3 of this Regulation. However, the energy banked during off-peak TOD slots\nshall be permitted to draw during off-peak ToD slot only."
57,ि(cid:361)पुरा\nTripura,TSECL,रा(cid:475) ओपन ए(cid:411)ेस िविनयमो ंम(cid:336) िनिद(cid:330)(cid:700) नही ंहै\nNot Specified in State Open Access Regulations,None,None,None,None



Saved: banking_charge_merged_preview.csv


In [ ]:
# ==========================================================
# PHASE 3A
# HIERARCHICAL ROW NORMALIZER
# ==========================================================

import pandas as pd

# ----------------------------------------------------------
# Load merged CSV
# ----------------------------------------------------------

df = pd.read_csv("banking_charge_merged_preview.csv")

# ----------------------------------------------------------
# Forward-fill state-level information
# ----------------------------------------------------------

state_col = 0

current_state = None

normalized_rows = []

for _, row in df.iterrows():

    row = row.copy()

    state = row[state_col]

    # ----------------------------------------------
    # new state encountered
    # ----------------------------------------------

    if pd.notna(state) and str(state).strip() != "":
        current_state = state

    # ----------------------------------------------
    # inherit state
    # ----------------------------------------------

    row[state_col] = current_state

    normalized_rows.append(row)

normalized_df = pd.DataFrame(normalized_rows)

print("="*80)
print("NORMALIZED SHAPE")
print(normalized_df.shape)
print("="*80)

display(normalized_df.head(25))

print("\n\nLAST 20 ROWS\n")
display(normalized_df.tail(20))

# ----------------------------------------------------------
# Save
# ----------------------------------------------------------

normalized_df.to_csv(
    "banking_charge_normalized.csv",
    index=False
)

print("\nSaved: banking_charge_normalized.csv")

NORMALIZED SHAPE
(63, 7)


/tmp/ipykernel_438/3731831226.py:28: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  state = row[state_col]
/tmp/ipykernel_438/3731831226.py:41: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  row[state_col] = current_state


,0,1,2,3,4,5,6
0,अंडमान और\nिनकोबार (cid:554)ीप\nसमूह\nAndaman &\nNicobar Islands,ED-A&NI,8.00%,Monthly/\nBilling\nCycle,NaN,NaN,"1. नवीकरणीय ऊजा(cid:330) उ(cid:523)ादन क(cid:336)(cid:363) (cid:554)ारा िकए गए कुल उ(cid:523)ादन के 20% तक सीिमत: शेष बची िवद्युत,\nकी 'डी(cid:817)ड खरीद' मानी जाएगी, िजसका भुगतान औसत िवद्यु(cid:5..."
1,आं(cid:364) (cid:366)देश\nAndhra\nPradesh,APEPDCL,8.00%,Monthly/\nBilling\nCycle,NaN,NaN,(cid:356)ीन एनज(cid:334) ओपन ए(cid:411)ेस के उपभो(cid:389)ा िबिलंग अविध के दौरान िवतरण लाइस(cid:336)सधारी से अपनी कुल मािसक\nिबजली खपत का केवल तीस (cid:366)ितशत (30%) ही जमा कर सक(cid:336)गे। 30% ...
2,आं(cid:364) (cid:366)देश\nAndhra\nPradesh,APSPDCL,NaN,NaN,NaN,NaN,NaN
3,आं(cid:364) (cid:366)देश\nAndhra\nPradesh,APCPDCL,NaN,NaN,NaN,NaN,NaN
4,आं(cid:364) (cid:366)देश\nAndhra\nPradesh,NaN,NaN,NaN,NaN,NaN,"Provided further that, the lapsed unutilized surplus banked energy enlisted for RECs, if not\nclaimed by the renewable energy generating station, the DISCOMs shall account for such\nlapsed energy ..."
5,अ(cid:348)णाचल (cid:366)देश\nArunachal\nPradesh,DoP,8.00%,Monthly/\nBilling\nCycle,NaN,NaN,"(cid:356)ीन एनज(cid:334) ओपन ए(cid:411)ेस उपभो(cid:389)ाओ ंके िलए ब(cid:339)िकंग की गई ऊजा(cid:330) की अनुमत मा(cid:361)ा , उपभो(cid:389)ा (cid:554)ारा िवतरण\nलाइस(cid:336)सधारी से की गई कुल मािसक..."
6,असम\nAssam,APDCL,8.00%,Billing\nCycle,NaN,NaN,"(cid:366)(cid:529)ेक िबिलंग आवत(cid:330)न के अंत म(cid:336), समा(cid:593) माना जाएगा और नवीकरणीय ऊजा(cid:330) उ(cid:523)ादन क(cid:336)(cid:363) उस सीमा तक\nनवीकरणीय ऊजा(cid:330) (cid:366)माणप(cid:..."
7,िबहार\nBihar,NBPDCL,"8%\nDuring Apr to Oct, drawl\nof banked energy during\npeak TOD period shall\nnot be permitted.",Monthly,NaN,NaN,"(cid:366)(cid:529)ेक िबिलंग आवत(cid:330)न के अंत म(cid:336), समा(cid:593) माना जाएगा और नवीकरणीय ऊजा(cid:330) उ(cid:523)ादन क(cid:336)(cid:363) उस सीमा तक\nनवीकरणीय ऊजा(cid:330) (cid:366)माणप(cid:..."
8,िबहार\nBihar,SBPDCL,NaN,NaN,NaN,NaN,NaN
9,चंडीगढ़\nChandigarh,CPDL,8.00%,Monthly/\nBilling\nCycle,NaN,NaN,"1. नवीकरणीय ऊजा(cid:330) उ(cid:523)ादन क(cid:336)(cid:363) (cid:554)ारा िकए गए कुल उ(cid:523)ादन के 20% तक सीिमत: शेष बची िवद्युत, की\n'डी(cid:817)ड खरीद' मानी जाएगी, िजसका भुगतान औसत िवद्युत् खरी..."




LAST 20 ROWS



,0,1,2,3,4,5,6
43,sनागाल(cid:339)ड\nNagaland,DoP-Nagaland,8.00%,Monthly,NaN,NaN,"(cid:366)(cid:529)ेक ब(cid:339)िकंग आवत(cid:330)न के अंत म(cid:336), उपयोग न की गई अिधशेष ब(cid:339)क की गई ऊजा(cid:330) को समा(cid:593) माना जाएगा;\nबशत(cid:337) िक, नवीकरणीय ऊजा(cid:330) उ(cid:5..."
44,ओिडशा\nOrissa,All DISCOMS,Solar = 8%\nWind or Solar and\nWind Hybrid = 5%,Monthly,NaN,NaN,"बशत(cid:337) िक, ब(cid:339)िकंग आवत(cid:330)न की समा(cid:304)(cid:593) पर उपयोग न की गई ब(cid:339)क की गई ऊजा(cid:330) को ि(cid:356)डको (cid:554)ारा 'डी(cid:817)ड खरीद'\nमाना जाएगा। इसकी गणना िपछल..."
45,ओिडशा\nOrissa,NaN,NaN,NaN,NaN,NaN,Provided that the unutilized banked energy would be treated as deemed purchase by\nGRIDCO on the expiry of banking cycle at lowest RE procurement price of GRIDCO of last\nfinancial year or lowest ...
46,पुडुचेरी\nPuducherry,ED-Puducherry,8.00%,Monthly/\nBilling\nCycle,NaN,NaN,"1. नवीकरणीय ऊजा(cid:330) उ(cid:523)ादन क(cid:336)(cid:363) (cid:554)ारा िकए गए कुल उ(cid:523)ादन के 20% तक सीिमत: शेष बची िवद्युत,\nकी 'डी(cid:817)ड खरीद' मानी जाएगी, िजसका भुगतान औसत िवद्युत ्खरी..."
47,पंजाब\nPunjab,PSPCL,Yet to decide,12\nmonths\n(FY),NaN,NaN,िव(cid:517) वष(cid:330) के अंत तक महीने-दर-महीने के आधार पर आगे बढ़ने की अनुमित।\nिव(cid:517) वष(cid:330) के अंत म(cid:336) अ(cid:366)यु(cid:389) ऊजा(cid:330) समा(cid:593) हो जाएगी और कोई मुआवजा नह...
48,राज(cid:725)ान\nRajasthan,AVVNL,8.00%,Monthly/\nBilling\nCycle,NaN,NaN,"िबिलंग आवत(cid:330)न की समा(cid:304)(cid:593) पर उपयोग न की गई ब(cid:339)क की गई ऊजा(cid:330) समा(cid:593) हो जाएगी और नवीकरणीय ऊजा(cid:330)\nकैि(cid:592)व उ(cid:523)ादन क(cid:336)(cid:363), िवद्य..."
49,राज(cid:725)ान\nRajasthan,JVVNL,NaN,NaN,NaN,NaN,Unutilized banked energy at the end of billing cycle shall lapse and the renewable energy\ncaptive generating plant shall be entitled to get Renewable Energy Certificates to the extent\nof the lap...
50,राज(cid:725)ान\nRajasthan,JdVVNL,NaN,NaN,NaN,NaN,NaN
51,िस(cid:304)(cid:383)म\nSikkim,Power Department,8.00%,Monthly,NaN,NaN,"महीने के अंत म(cid:336) उपयोग न की गई अिधशेष ब(cid:339)क की गई ऊजा(cid:330) को (cid:366)(cid:529)ेक ब(cid:339)िकंग आवत(cid:330)न के अंत म(cid:336) समा(cid:593)\nमाना जाएगा; बशत(cid:337) िक, नवीकरण..."
52,तिमलनाडु\nTamil Nadu,TNPDCL,8.00%,Monthly,NaN,NaN,ब(cid:339)िकंग अविध के अंत म(cid:336) अ(cid:366)यु(cid:389) अिधशेष ब(cid:339)(cid:779)ड ऊजा(cid:330) को िवतरण लाइस(cid:336)सधारी को आयोग के आदेशो ंके\nअनुसार लागू संबंिधत आरई दरो ंके 75% की दर पर ...



Saved: banking_charge_normalized.csv


In [ ]:
# ==========================================================
# PHASE 3B
# CLASSIFY RECORD ROWS VS CONTINUATION ROWS
# ==========================================================

import pandas as pd

df = pd.read_csv("banking_charge_normalized.csv")

record_rows = []
continuation_rows = []

for idx, row in df.iterrows():

    state = str(row.iloc[0]).strip() if pd.notna(row.iloc[0]) else ""
    discom = str(row.iloc[1]).strip() if pd.notna(row.iloc[1]) else ""

    charge = str(row.iloc[2]).strip() if pd.notna(row.iloc[2]) else ""
    period = str(row.iloc[3]).strip() if pd.notna(row.iloc[3]) else ""

    # ---------------------------------------------
    # Record row
    # ---------------------------------------------

    if discom != "" and discom.lower() != "nan":

        record_rows.append({
            "idx": idx,
            "state": state,
            "discom": discom,
            "charge_present": charge != "" and charge.lower() != "nan",
            "period_present": period != "" and period.lower() != "nan"
        })

    # ---------------------------------------------
    # Continuation row
    # ---------------------------------------------

    else:

        continuation_rows.append({
            "idx": idx,
            "state": state
        })

print("="*80)
print("TOTAL ROWS:", len(df))
print("RECORD ROWS:", len(record_rows))
print("CONTINUATION ROWS:", len(continuation_rows))
print("="*80)

print("\nFIRST 20 RECORD ROWS\n")
display(pd.DataFrame(record_rows).head(20))

print("\nFIRST 20 CONTINUATION ROWS\n")
display(pd.DataFrame(continuation_rows).head(20))

TOTAL ROWS: 63
RECORD ROWS: 55
CONTINUATION ROWS: 8

FIRST 20 RECORD ROWS



,idx,state,discom,charge_present,period_present
0,0,अंडमान और\nिनकोबार (cid:554)ीप\nसमूह\nAndaman &\nNicobar Islands,ED-A&NI,True,True
1,1,आं(cid:364) (cid:366)देश\nAndhra\nPradesh,APEPDCL,True,True
2,2,आं(cid:364) (cid:366)देश\nAndhra\nPradesh,APSPDCL,False,False
3,3,आं(cid:364) (cid:366)देश\nAndhra\nPradesh,APCPDCL,False,False
4,5,अ(cid:348)णाचल (cid:366)देश\nArunachal\nPradesh,DoP,True,True
5,6,असम\nAssam,APDCL,True,True
6,7,िबहार\nBihar,NBPDCL,True,True
7,8,िबहार\nBihar,SBPDCL,False,False
8,9,चंडीगढ़\nChandigarh,CPDL,True,True
9,11,छ(cid:517)ीसगढ\nChhattisgarh,CSPDCL,True,True



FIRST 20 CONTINUATION ROWS



,idx,state
0,4,आं(cid:364) (cid:366)देश\nAndhra\nPradesh
1,10,चंडीगढ़\nChandigarh
2,39,महारा(cid:700)(cid:332)\nMaharashtra
3,45,ओिडशा\nOrissa
4,53,तिमलनाडु\nTamil Nadu
5,56,तेलंगाना\nTelangana
6,59,उ(cid:517)र (cid:366)देश\nUttar Pradesh
7,60,उ(cid:517)र (cid:366)देश\nUttar Pradesh


In [ ]:
# ==========================================================
# PHASE 3C
# CREATE FULLY NORMALIZED DISCOM RECORDS
# ==========================================================

import pandas as pd
import json

df = pd.read_csv("banking_charge_normalized.csv")

records = []

current_master = None

for _, row in df.iterrows():

    state = row.iloc[0]
    discom = row.iloc[1]
    charge = row.iloc[2]
    period = row.iloc[3]
    policy = row.iloc[6]

    state = "" if pd.isna(state) else str(state).strip()
    discom = "" if pd.isna(discom) else str(discom).strip()
    charge = "" if pd.isna(charge) else str(charge).strip()
    period = "" if pd.isna(period) else str(period).strip()
    policy = "" if pd.isna(policy) else str(policy).strip()

    # ------------------------------------------------------
    # TYPE A
    # New master record
    # ------------------------------------------------------

    if discom and charge:

        current_master = {
            "state": state,
            "discom": discom,
            "charge": charge,
            "period": period,
            "policy": policy
        }

        records.append(current_master)

    # ------------------------------------------------------
    # TYPE B
    # Child DISCOM inherits parent values
    # ------------------------------------------------------

    elif discom and not charge:

        if current_master:

            records.append({
                "state": state,
                "discom": discom,
                "charge": current_master["charge"],
                "period": current_master["period"],
                "policy": current_master["policy"]
            })

    # ------------------------------------------------------
    # TYPE C
    # Continuation row
    # ------------------------------------------------------

    elif not discom:

        if records and policy:

            records[-1]["policy"] += "\n" + policy

# ----------------------------------------------------------
# Save
# ----------------------------------------------------------

with open(
    "banking_charge_normalized_records.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        records,
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 80)
print("TOTAL NORMALIZED RECORDS:", len(records))
print("=" * 80)

print("\nFIRST 10 RECORDS\n")

for r in records[:10]:
    print(json.dumps(r, indent=2, ensure_ascii=False))
    print("-" * 50)

print("\nSaved:")
print("banking_charge_normalized_records.json")

TOTAL NORMALIZED RECORDS: 55

FIRST 10 RECORDS

{
  "state": "अंडमान और\nिनकोबार (cid:554)ीप\nसमूह\nAndaman &\nNicobar Islands",
  "discom": "ED-A&NI",
  "charge": "8.00%",
  "period": "Monthly/\nBilling\nCycle",
  "policy": "1. नवीकरणीय ऊजा(cid:330) उ(cid:523)ादन क(cid:336)(cid:363) (cid:554)ारा िकए गए कुल उ(cid:523)ादन के 20% तक सीिमत: शेष बची िवद्युत,\nकी 'डी(cid:817)ड खरीद' मानी जाएगी, िजसका भुगतान औसत िवद्यु(cid:514)रीद लागत (एपीपीसी) या उस वष(cid:330) के िलए\nिनधा(cid:330)(cid:303)रत फीड-इन-टै(cid:303)रफ (िबना स(cid:304)(cid:629)डी और (cid:533)(cid:303)रत मू(cid:670)(cid:376)ास के लाभ को जोड़े), दोनो ंम(cid:336) से जो भी कम\nहो, उस दर पर िकया जाएगा।\nLimited to 30% of the total generation by RE Generating Station: Deemed purchase at\nAverage Power Purchase Cost (APPC) or Feed-in-Tariff determined for that year without\nconsidering subsidy and Accelerated Depreciation, whichever is lower.\n2. नवीकरणीय ऊजा(cid:330) उ(cid:523)ादन क(cid:336)(cid:363) (cid:554)ारा िकए गए कुल उ(cid:523)

In [ ]:
# ==========================================================
# PHASE 3D
# DISCOVER MASTER-CHILD GROUPS
# ==========================================================

import pandas as pd

df = pd.read_csv("banking_charge_normalized.csv")

groups = []

current_group = None

for idx, row in df.iterrows():

    state = "" if pd.isna(row.iloc[0]) else str(row.iloc[0]).strip()
    discom = "" if pd.isna(row.iloc[1]) else str(row.iloc[1]).strip()
    charge = "" if pd.isna(row.iloc[2]) else str(row.iloc[2]).strip()

    # ---------------------------------------------
    # New master record
    # ---------------------------------------------

    if discom and charge:

        current_group = {
            "state": state,
            "master_discom": discom,
            "children": [discom]
        }

        groups.append(current_group)

    # ---------------------------------------------
    # Child discom
    # ---------------------------------------------

    elif discom and not charge:

        if current_group:
            current_group["children"].append(discom)

print("=" * 80)
print("TOTAL GROUPS:", len(groups))
print("=" * 80)

for g in groups[:20]:

    print("\nSTATE:", g["state"])
    print("COUNT:", len(g["children"]))
    print("DISCOMS:", g["children"])

TOTAL GROUPS: 35

STATE: अंडमान और
िनकोबार (cid:554)ीप
समूह
Andaman &
Nicobar Islands
COUNT: 1
DISCOMS: ['ED-A&NI']

STATE: आं(cid:364) (cid:366)देश
Andhra
Pradesh
COUNT: 3
DISCOMS: ['APEPDCL', 'APSPDCL', 'APCPDCL']

STATE: अ(cid:348)णाचल (cid:366)देश
Arunachal
Pradesh
COUNT: 1
DISCOMS: ['DoP']

STATE: असम
Assam
COUNT: 1
DISCOMS: ['APDCL']

STATE: िबहार
Bihar
COUNT: 2
DISCOMS: ['NBPDCL', 'SBPDCL']

STATE: चंडीगढ़
Chandigarh
COUNT: 1
DISCOMS: ['CPDL']

STATE: छ(cid:517)ीसगढ
Chhattisgarh
COUNT: 1
DISCOMS: ['CSPDCL']

STATE: दादरा और नगर
हवेली और दमन
और दीव
Dadra & Nagar
Haveli and
Daman & Diu
COUNT: 1
DISCOMS: ['DNHDDPDCL']

STATE: िद(cid:671)ी
Delhi
COUNT: 3
DISCOMS: ['BRPL', 'BYPL', 'TPDDL']

STATE: गोवा
Goa
COUNT: 1
DISCOMS: ['ED-Goa']

STATE: गुजरात
Gujarat
COUNT: 4
DISCOMS: ['DGVCL', 'MGVCL', 'PGVCL', 'UGVCL']

STATE: ह(cid:303)रयाणा
Haryana
COUNT: 2
DISCOMS: ['DHBVNL', 'UHBVNL']

STATE: िहमाचल (cid:366)देश
Himachal
Pradesh
COUNT: 1
DISCOMS: ['HPSEBL']

STATE: ज(cid:643)ू एवं क(cid:6

In [ ]:
import pdfplumber

PDF_PATH = "/content/co-ursi-key-regultry-parmtrs-pwr-utiltis-dt180526.pdf"

for p in [62, 63]:

    page = pdfplumber.open(PDF_PATH).pages[p-1]

    tables = page.extract_tables()

    print("\nPAGE:", p)
    print("TABLES:", len(tables))

    for i, t in enumerate(tables):

        rows = len(t)
        cols = max(len(r) for r in t if r)

        print(
            f"table={i} rows={rows} cols={cols}"
        )


PAGE: 62
TABLES: 6
table=0 rows=32 cols=18
table=1 rows=3 cols=1
table=2 rows=3 cols=1
table=3 rows=2 cols=1
table=4 rows=2 cols=1
table=5 rows=2 cols=1

PAGE: 63
TABLES: 6
table=0 rows=33 cols=18
table=1 rows=3 cols=1
table=2 rows=3 cols=1
table=3 rows=2 cols=1
table=4 rows=2 cols=1
table=5 rows=2 cols=1


In [ ]:
# ==========================================================
# PHASE 3E
# TEXT NORMALIZATION LAYER
# ==========================================================

import pandas as pd
import re

df = pd.read_csv("banking_charge_normalized.csv")

# ----------------------------------------------------------
# Remove PDF garbage
# ----------------------------------------------------------

def clean_pdf_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    # remove cid artifacts
    text = re.sub(r"\(cid:\d+\)", " ", text)

    # normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# ----------------------------------------------------------
# Extract English state name
# ----------------------------------------------------------

def extract_state_name(state_text):

    state_text = clean_pdf_text(state_text)

    lines = state_text.split()

    english_tokens = []

    for token in lines:

        # keep tokens containing English letters
        if re.search(r"[A-Za-z]", token):
            english_tokens.append(token)

    return " ".join(english_tokens)


# ----------------------------------------------------------
# Apply cleaning
# ----------------------------------------------------------

df["state_clean"] = df.iloc[:,0].apply(extract_state_name)

df["discom_clean"] = (
    df.iloc[:,1]
    .fillna("")
    .astype(str)
    .str.replace("\n"," ", regex=False)
    .str.strip()
)

# charge column
df["charge_clean"] = (
    df.iloc[:,2]
    .fillna("")
    .astype(str)
    .str.replace("\n"," ", regex=False)
)

# period column
df["period_clean"] = (
    df.iloc[:,3]
    .fillna("")
    .astype(str)
    .str.replace("\n"," ", regex=False)
)

# policy column
df["policy_clean"] = (
    df.iloc[:,6]
    .fillna("")
    .apply(clean_pdf_text)
)

print("="*80)
print("STATE CLEANING PREVIEW")
print("="*80)

preview = df[
    [
        "state_clean",
        "discom_clean",
        "charge_clean",
        "period_clean"
    ]
].head(30)

display(preview)

# ----------------------------------------------------------
# Unique states discovered
# ----------------------------------------------------------

unique_states = (
    df["state_clean"]
    .dropna()
    .unique()
)

print("\nTOTAL UNIQUE STATES:", len(unique_states))

for s in sorted(unique_states):
    print(s)

# ----------------------------------------------------------
# Save
# ----------------------------------------------------------

df.to_csv(
    "banking_charge_cleaned.csv",
    index=False
)

print("\nSaved: banking_charge_cleaned.csv")

STATE CLEANING PREVIEW


,state_clean,discom_clean,charge_clean,period_clean
0,Andaman Nicobar Islands,ED-A&NI,8.00%,Monthly/ Billing Cycle
1,Andhra Pradesh,APEPDCL,8.00%,Monthly/ Billing Cycle
2,Andhra Pradesh,APSPDCL,,
3,Andhra Pradesh,APCPDCL,,
4,Andhra Pradesh,,,
5,Arunachal Pradesh,DoP,8.00%,Monthly/ Billing Cycle
6,Assam,APDCL,8.00%,Billing Cycle
7,Bihar,NBPDCL,"8% During Apr to Oct, drawl of banked energy during peak TOD period shall not be permitted.",Monthly
8,Bihar,SBPDCL,,
9,Chandigarh,CPDL,8.00%,Monthly/ Billing Cycle



TOTAL UNIQUE STATES: 36
*West Bengal
Andaman Nicobar Islands
Andhra Pradesh
Arunachal Pradesh
Assam
Bihar
Chandigarh
Chhattisgarh
Dadra Nagar Haveli and Daman Diu
Delhi
Goa
Gujarat
Haryana
Himachal Pradesh
J&K
Jharkhand
Karnataka
Kerala
Ladakh
Lakshadweep
Madhya Pradesh
Maharashtra
Manipur
Meghalaya
Mizoram
Orissa
Puducherry
Punjab
Rajasthan
Sikkim
Tamil Nadu
Telangana
Tripura
Uttar Pradesh
Uttarakhand
sनागाल Nagaland

Saved: banking_charge_cleaned.csv


In [ ]:
# ==========================================================
# PHASE 4A
# UNIVERSAL PARAMETER EXTRACTION ENGINE
# ==========================================================

import pdfplumber
import pandas as pd
import re
import json

# ==========================================================
# TEXT CLEANING
# ==========================================================

def clean_pdf_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    text = re.sub(r"\(cid:\d+\)", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def extract_state_name(state_text):

    state_text = clean_pdf_text(state_text)

    tokens = state_text.split()

    english_tokens = []

    for token in tokens:

        if re.search(r"[A-Za-z]", token):
            english_tokens.append(token)

    state = " ".join(english_tokens)

    # manual cleanup rules
    replacements = {
        "*West Bengal": "West Bengal",
        "Orissa": "Odisha",
        "s Nagaland": "Nagaland",
        "Nagaland": "Nagaland"
    }

    state = replacements.get(state, state)

    return state


# ==========================================================
# CORE ENGINE
# ==========================================================

def extract_parameter_family(
    pdf_path,
    parameter_name,
    start_page,
    end_page
):

    print("=" * 80)
    print("EXTRACTING:", parameter_name)
    print("PAGES:", start_page, "->", end_page)
    print("=" * 80)

    # ------------------------------------------------------
    # STAGE 1
    # extract largest table per page
    # ------------------------------------------------------

    merged_rows = []

    with pdfplumber.open(pdf_path) as pdf:

        for page_num in range(start_page, end_page + 1):

            page = pdf.pages[page_num - 1]

            tables = page.extract_tables()

            if not tables:
                continue

            largest_table = max(
                tables,
                key=lambda t: len(t) * max(len(r) for r in t if r)
            )

            if len(largest_table) < 2:
                continue

            data_rows = largest_table[4:]

            merged_rows.extend(data_rows)

    # ------------------------------------------------------
    # STAGE 2
    # dataframe
    # ------------------------------------------------------

    df = pd.DataFrame(merged_rows)

    # ------------------------------------------------------
    # STAGE 3
    # state inheritance
    # ------------------------------------------------------

    current_state = None

    for idx in range(len(df)):

        state = df.iloc[idx, 0]

        if pd.notna(state) and str(state).strip():

            current_state = state

        else:

            df.iloc[idx, 0] = current_state

    # ------------------------------------------------------
    # STAGE 4
    # normalized records
    # ------------------------------------------------------

    records = []

    current_master = None

    for _, row in df.iterrows():

        state = extract_state_name(row.iloc[0])

        discom = clean_pdf_text(row.iloc[1])

        charge = clean_pdf_text(row.iloc[2])

        period = clean_pdf_text(row.iloc[3])

        policy = ""

        if len(row) > 6:
            policy = clean_pdf_text(row.iloc[6])

        # ----------------------------------------------
        # master row
        # ----------------------------------------------

        if discom and charge:

            current_master = {
                "state": state,
                "discom": discom,
                "charge": charge,
                "period": period,
                "policy": policy
            }

            records.append(current_master)

        # ----------------------------------------------
        # child row
        # ----------------------------------------------

        elif discom and not charge:

            if current_master:

                records.append({
                    "state": state,
                    "discom": discom,
                    "charge": current_master["charge"],
                    "period": current_master["period"],
                    "policy": current_master["policy"]
                })

        # ----------------------------------------------
        # continuation row
        # ----------------------------------------------

        elif policy:

            if records:

                records[-1]["policy"] += "\n" + policy

    # ------------------------------------------------------
    # result
    # ------------------------------------------------------

    result = {
        "parameter": parameter_name,
        "record_count": len(records),
        "records": records
    }

    return result


# ==========================================================
# TEST 1
# BANKING CHARGES
# ==========================================================

PDF_PATH = "/content/co-ursi-key-regultry-parmtrs-pwr-utiltis-dt180526.pdf"

banking_result = extract_parameter_family(
    PDF_PATH,
    "BANKING CHARGES",
    64,
    75
)

print("\nBANKING RECORDS:", banking_result["record_count"])

print("\nFIRST 5 RECORDS\n")

for r in banking_result["records"][:5]:

    print(json.dumps(r, indent=2, ensure_ascii=False))
    print("-" * 60)

# ==========================================================
# SAVE
# ==========================================================

with open(
    "banking_charge_final.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        banking_result,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nSaved: banking_charge_final.json")

EXTRACTING: BANKING CHARGES
PAGES: 64 -> 75

BANKING RECORDS: 55

FIRST 5 RECORDS

{
  "state": "Andaman Nicobar Islands",
  "discom": "ED-A&NI",
  "charge": "8.00%",
  "period": "Monthly/ Billing Cycle",
  "policy": "1. नवीकरणीय ऊजा उ ादन क ारा िकए गए कुल उ ादन के 20% तक सीिमत: शेष बची िवद्युत, की 'डी ड खरीद' मानी जाएगी, िजसका भुगतान औसत िवद्यु रीद लागत (एपीपीसी) या उस वष के िलए िनधा रत फीड-इन-टै रफ (िबना स डी और रत मू ास के लाभ को जोड़े), दोनो ंम से जो भी कम हो, उस दर पर िकया जाएगा। Limited to 30% of the total generation by RE Generating Station: Deemed purchase at Average Power Purchase Cost (APPC) or Feed-in-Tariff determined for that year without considering subsidy and Accelerated Depreciation, whichever is lower. 2. नवीकरणीय ऊजा उ ादन क ारा िकए गए कुल उ ादन के 20% से अिधक होने पर: यह ऊजा तः समा हो जाएगी और इसके िलए कोई मुआवजा नही ंिदया जाएगा। In excess of 20% of the total generation by RE Generating Station: To lapse and no compensation."
}
---------------------------------------

In [ ]:
transmission_result = extract_parameter_family(
    PDF_PATH,
    "TRANSMISSION CHARGE",
    62,
    63
)

print("\nRECORD COUNT:")
print(transmission_result["record_count"])

print("\nFIRST 10 RECORDS")

for r in transmission_result["records"][:10]:
    print(json.dumps(r, indent=2, ensure_ascii=False))
    print("-"*60)

EXTRACTING: TRANSMISSION CHARGE
PAGES: 62 -> 63

RECORD COUNT:
0

FIRST 10 RECORDS


In [ ]:
import pdfplumber
import pandas as pd

PDF_PATH = "/content/co-ursi-key-regultry-parmtrs-pwr-utiltis-dt180526.pdf"

page = pdfplumber.open(PDF_PATH).pages[61]   # page 62

tables = page.extract_tables()

largest = max(
    tables,
    key=lambda t: len(t) * max(len(r) for r in t if r)
)

df = pd.DataFrame(largest)

print("Shape:", df.shape)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

display(df.head(15))

Shape: (32, 18)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,रा(cid:475)/क(cid:336)(cid:363) शािसत (cid:366)देश /\nStates/UTs,None,None,लागू वष(cid:330)\nApplicable\nYear,None,None,,दीघ(cid:330)कािलक एवं,,इकाई\nUnit,None,None,अ(cid:665)कािलक\nShort\nTerm,None,None,इकाई\nUnit,None,None
1,None,None,None,None,None,None,None,म(cid:559)म अविध,None,None,None,None,None,None,None,None,None,None
2,None,None,None,None,None,None,None,Long &,None,None,None,None,None,None,None,None,None,None
3,None,None,None,None,None,None,None,Medium,None,None,None,None,None,None,None,None,None,None
4,None,None,None,None,None,None,None,Term,None,None,None,None,None,None,None,None,None,None
5,आं(cid:364) (cid:366)देश / Andhra Pradesh*,None,None,2026-27,None,None,201.80,None,None,Rs/kVA/month,None,None,201.80,None,None,Rs/kVA/month,None,None
6,एपीट(cid:332)ान(cid:715)ो / APTRANSCO,None,None,2026-27,None,None,201.80,None,None,Rs/kVA/month,None,None,201.80,None,None,Rs/kVA/month,None,None
7,अ(cid:348)णाचल (cid:366)देश / Arunachal\nPradesh,None,None,2023-24,None,None,"71,253",None,None,Rs/MW/month,None,None,0,None,None,Rs/kWh,None,None
8,अ(cid:348)णाचल पीडी / Arunachal PD,None,None,2023-24,None,None,"71,253",None,None,Rs/MW/month,None,None,0,None,None,Rs/kWh,None,None
9,असम / Assam*,None,None,2026-27,None,None,--,None,None,--,None,None,0.45,None,None,Rs/kWh,None,None


In [ ]:
import pandas as pd
import pdfplumber

PDF_PATH = "/content/co-ursi-key-regultry-parmtrs-pwr-utiltis-dt180526.pdf"

page = pdfplumber.open(PDF_PATH).pages[61]

largest = max(
    page.extract_tables(),
    key=lambda t: len(t) * max(len(r) for r in t if r)
)

df = pd.DataFrame(largest)

print("Shape:", df.shape)

for col in df.columns:

    non_null = df[col].notna().sum()

    if non_null > 0:
        print(
            f"COL {col}: {non_null} non-null"
        )

Shape: (32, 18)
COL 0: 28 non-null
COL 1: 2 non-null
COL 2: 2 non-null
COL 3: 28 non-null
COL 4: 4 non-null
COL 5: 4 non-null
COL 6: 28 non-null
COL 7: 9 non-null
COL 8: 5 non-null
COL 9: 28 non-null
COL 10: 3 non-null
COL 11: 3 non-null
COL 12: 28 non-null
COL 13: 3 non-null
COL 14: 3 non-null
COL 15: 28 non-null
COL 16: 3 non-null
COL 17: 3 non-null


In [ ]:
# ==========================================================
# PHASE 5A
# NUMERIC MATRIX PARSER
# TRANSMISSION CHARGES
# ==========================================================

import pdfplumber
import pandas as pd
import json
import re

PDF_PATH = "/content/co-ursi-key-regultry-parmtrs-pwr-utiltis-dt180526.pdf"

# ----------------------------------------------------------
# CLEANERS
# ----------------------------------------------------------

def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x)

    x = re.sub(r"\(cid:\d+\)", " ", x)
    x = re.sub(r"\s+", " ", x)

    return x.strip()


def extract_english(text):

    text = clean_text(text)

    tokens = []

    for t in text.split():

        if re.search(r"[A-Za-z]", t):
            tokens.append(t)

    return " ".join(tokens)


# ----------------------------------------------------------
# EXTRACT TRANSMISSION TABLE
# ----------------------------------------------------------

all_rows = []

with pdfplumber.open(PDF_PATH) as pdf:

    for page_num in [62, 63]:

        page = pdf.pages[page_num - 1]

        tables = page.extract_tables()

        largest = max(
            tables,
            key=lambda t: len(t) * max(len(r) for r in t if r)
        )

        all_rows.extend(largest)

df = pd.DataFrame(all_rows)

print("Merged Shape:", df.shape)

# ----------------------------------------------------------
# REMOVE HEADER ROWS
# ----------------------------------------------------------

data = df.iloc[5:].reset_index(drop=True)

print("Data Shape:", data.shape)

# ----------------------------------------------------------
# ACTIVE COLS DISCOVERED
# ----------------------------------------------------------

STATE_COL = 0
YEAR_COL = 3

LONG_CHARGE_COL = 6
LONG_UNIT_COL = 9

SHORT_CHARGE_COL = 12
SHORT_UNIT_COL = 15

# ----------------------------------------------------------
# PARSE
# ----------------------------------------------------------

records = []

current_state = None

for _, row in data.iterrows():

    raw_name = clean_text(row[STATE_COL])

    year = clean_text(row[YEAR_COL])

    long_charge = clean_text(row[LONG_CHARGE_COL])
    long_unit = clean_text(row[LONG_UNIT_COL])

    short_charge = clean_text(row[SHORT_CHARGE_COL])
    short_unit = clean_text(row[SHORT_UNIT_COL])

    name = extract_english(raw_name)

    if not name:
        continue

    # ---------------------------------------------
    # detect STATE row
    # ---------------------------------------------

    if any(
        keyword in name
        for keyword in [
            "Pradesh",
            "Delhi",
            "Goa",
            "Gujarat",
            "Bihar",
            "Assam",
            "Jharkhand",
            "Punjab",
            "Odisha",
            "Tamil",
            "Kerala",
            "Karnataka",
            "Haryana",
            "Maharashtra",
            "Nagaland",
            "Tripura",
            "Sikkim",
            "Mizoram",
            "Meghalaya",
            "Manipur",
            "Uttarakhand",
            "Bengal",
            "J&K",
            "Ladakh"
        ]
    ):

        current_state = name

    # ---------------------------------------------
    # utility row
    # ---------------------------------------------

    else:

        records.append({
            "state": current_state,
            "utility": name,
            "year": year,
            "long_medium_charge": long_charge,
            "long_medium_unit": long_unit,
            "short_term_charge": short_charge,
            "short_term_unit": short_unit
        })

# ----------------------------------------------------------
# RESULTS
# ----------------------------------------------------------

print("=" * 80)
print("TOTAL RECORDS:", len(records))
print("=" * 80)

print("\nFIRST 15 RECORDS\n")

for r in records[:15]:

    print(json.dumps(r, indent=2))
    print("-" * 50)

# ----------------------------------------------------------
# SAVE
# ----------------------------------------------------------

with open(
    "transmission_charge_records.json",
    "w"
) as f:

    json.dump(
        records,
        f,
        indent=2
    )

print("\nSaved: transmission_charge_records.json")

Merged Shape: (65, 18)
Data Shape: (60, 18)
TOTAL RECORDS: 30

FIRST 15 RECORDS

{
  "state": "Andhra Pradesh*",
  "utility": "APTRANSCO",
  "year": "2026-27",
  "long_medium_charge": "201.80",
  "long_medium_unit": "Rs/kVA/month",
  "short_term_charge": "201.80",
  "short_term_unit": "Rs/kVA/month"
}
--------------------------------------------------
{
  "state": "Arunachal Pradesh",
  "utility": "Arunachal PD",
  "year": "2023-24",
  "long_medium_charge": "71,253",
  "long_medium_unit": "Rs/MW/month",
  "short_term_charge": "0",
  "short_term_unit": "Rs/kWh"
}
--------------------------------------------------
{
  "state": "Assam*",
  "utility": "AEGCL",
  "year": "2026-27",
  "long_medium_charge": "--",
  "long_medium_unit": "--",
  "short_term_charge": "0.45",
  "short_term_unit": "Rs/kWh"
}
--------------------------------------------------
{
  "state": "Bihar",
  "utility": "BSPTCL",
  "year": "",
  "long_medium_charge": "",
  "long_medium_unit": "",
  "short_term_charge": "",
  

In [ ]:
import pdfplumber
import pandas as pd
import re

PDF_PATH = "/content/co-ursi-key-regultry-parmtrs-pwr-utiltis-dt180526.pdf"

page = pdfplumber.open(PDF_PATH).pages[61]

largest = max(
    page.extract_tables(),
    key=lambda t: len(t) * max(len(r) for r in t if r)
)

df = pd.DataFrame(largest)

data = df.iloc[5:].reset_index(drop=True)

def clean(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = re.sub(r"\(cid:\d+\)", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()

print("="*100)

for i in range(min(25, len(data))):

    name = clean(data.iloc[i,0])

    year = clean(data.iloc[i,3])

    lm = clean(data.iloc[i,6])

    st = clean(data.iloc[i,12])

    print(
        f"{i:02d} | "
        f"name={name[:50]} | "
        f"year={year} | "
        f"LM={lm} | "
        f"ST={st}"
    )

00 | name=आं देश / Andhra Pradesh* | year=2026-27 | LM=201.80 | ST=201.80
01 | name=एपीट ान ो / APTRANSCO | year=2026-27 | LM=201.80 | ST=201.80
02 | name=अ णाचल देश / Arunachal Pradesh | year=2023-24 | LM=71,253 | ST=0
03 | name=अ णाचल पीडी / Arunachal PD | year=2023-24 | LM=71,253 | ST=0
04 | name=असम / Assam* | year=2026-27 | LM=-- | ST=0.45
05 | name=एईजीसीएल / AEGCL | year=2026-27 | LM=-- | ST=0.45
06 | name=िबहार / Bihar | year=2026-27 | LM=-- | ST=--
07 | name=बीएसपीटीसीएल / BSPTCL | year= | LM= | ST=
08 | name=बीजीसीएल / BGCL | year= | LM= | ST=
09 | name=छ ीसगढ़ / Chhattisgarh* | year=2025-26 | LM=149.56 | ST=0.34
10 | name=सीएसपीटीसीएल / CSPTCL | year=2025-26 | LM=149.56 | ST=0.34
11 | name=दादरा और नगर हवेली और दमन और दीव Dadra & Nagar Hav | year=2026-27 | LM=98,573 | ST=3,241
12 | name=डीएनएचडीपीसीएल / DNHDDPCL | year=2026-27 | LM=98,573 | ST=3,241
13 | name=िद ी / Delhi | year=2021-22 | LM=92.86 | ST=N/A
14 | name=डीटीएल / DTL | year=2021-22 | LM=92.86 | ST=N/A
15 | name=गु

In [ ]:
# ==========================================================
# PHASE 6A
# GENERIC HIERARCHICAL TABLE PARSER
# ==========================================================

import pandas as pd
import re
import json

# ----------------------------------------------------------
# CLEANERS
# ----------------------------------------------------------

def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x)

    x = re.sub(r"\(cid:\d+\)", " ", x)
    x = re.sub(r"\s+", " ", x)

    return x.strip()


def extract_english(text):

    text = clean_text(text)

    # use text after "/" when available
    if "/" in text:
        text = text.split("/")[-1]

    text = text.strip()

    return text


# ==========================================================
# GENERIC PARSER
# ==========================================================

def parse_hierarchical_numeric_matrix(df):

    records = []

    current_parent = None

    # ---------------------------------------------
    # active columns discovered earlier
    # ---------------------------------------------

    NAME_COL = 0
    YEAR_COL = 3

    LONG_COL = 6
    LONG_UNIT_COL = 9

    SHORT_COL = 12
    SHORT_UNIT_COL = 15

    for idx, row in df.iterrows():

        name = extract_english(row[NAME_COL])

        if not name:
            continue

        year = clean_text(row[YEAR_COL])

        long_charge = clean_text(row[LONG_COL])
        long_unit = clean_text(row[LONG_UNIT_COL])

        short_charge = clean_text(row[SHORT_COL])
        short_unit = clean_text(row[SHORT_UNIT_COL])

        # -----------------------------------------
        # detect parent row
        #
        # utility rows are usually ALL CAPS
        # state rows usually contain spaces
        # -----------------------------------------

        looks_like_parent = (
            " " in name
            or "&" in name
            or "Pradesh" in name
            or "Delhi" in name
            or "Bihar" in name
            or "Assam" in name
            or "Goa" in name
            or "Gujarat" in name
            or "Punjab" in name
            or "Odisha" in name
            or "Jharkhand" in name
            or "Karnataka" in name
            or "Kerala" in name
            or "Haryana" in name
        )

        # -----------------------------------------
        # PARENT ROW
        # -----------------------------------------

        if looks_like_parent:

            current_parent = {
                "state": name,
                "year": year,
                "long_medium_charge": long_charge,
                "long_medium_unit": long_unit,
                "short_term_charge": short_charge,
                "short_term_unit": short_unit
            }

        # -----------------------------------------
        # CHILD UTILITY
        # -----------------------------------------

        else:

            if current_parent:

                records.append({
                    "state": current_parent["state"],
                    "utility": name,
                    "year": current_parent["year"],
                    "long_medium_charge":
                        current_parent["long_medium_charge"],
                    "long_medium_unit":
                        current_parent["long_medium_unit"],
                    "short_term_charge":
                        current_parent["short_term_charge"],
                    "short_term_unit":
                        current_parent["short_term_unit"]
                })

    return records


# ==========================================================
# LOAD TRANSMISSION DATAFRAME
# ==========================================================

import pdfplumber

PDF_PATH = "/content/co-ursi-key-regultry-parmtrs-pwr-utiltis-dt180526.pdf"

all_rows = []

with pdfplumber.open(PDF_PATH) as pdf:

    for page_num in [62,63]:

        page = pdf.pages[page_num - 1]

        tables = page.extract_tables()

        largest = max(
            tables,
            key=lambda t: len(t) * max(len(r) for r in t if r)
        )

        all_rows.extend(largest)

df = pd.DataFrame(all_rows)

df = df.iloc[5:].reset_index(drop=True)

# ==========================================================
# PARSE
# ==========================================================

records = parse_hierarchical_numeric_matrix(df)

print("="*80)
print("TOTAL RECORDS:", len(records))
print("="*80)

print("\nFIRST 20 RECORDS\n")

for r in records[:20]:

    print(
        json.dumps(
            r,
            indent=2,
            ensure_ascii=False
        )
    )
    print("-"*50)

# ==========================================================
# SAVE
# ==========================================================

with open(
    "transmission_charge_hierarchical.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        records,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nSaved:")
print("transmission_charge_hierarchical.json")

TOTAL RECORDS: 33

FIRST 20 RECORDS

{
  "state": "Andhra Pradesh*",
  "utility": "APTRANSCO",
  "year": "2026-27",
  "long_medium_charge": "201.80",
  "long_medium_unit": "Rs/kVA/month",
  "short_term_charge": "201.80",
  "short_term_unit": "Rs/kVA/month"
}
--------------------------------------------------
{
  "state": "Assam*",
  "utility": "AEGCL",
  "year": "2026-27",
  "long_medium_charge": "--",
  "long_medium_unit": "--",
  "short_term_charge": "0.45",
  "short_term_unit": "Rs/kWh"
}
--------------------------------------------------
{
  "state": "Bihar",
  "utility": "BSPTCL",
  "year": "2026-27",
  "long_medium_charge": "--",
  "long_medium_unit": "--",
  "short_term_charge": "--",
  "short_term_unit": "--"
}
--------------------------------------------------
{
  "state": "Bihar",
  "utility": "BGCL",
  "year": "2026-27",
  "long_medium_charge": "--",
  "long_medium_unit": "--",
  "short_term_charge": "--",
  "short_term_unit": "--"
}
-----------------------------------------

In [ ]:
# ==========================================================
# PHASE 6B
# BUILD STATE MASTER
# ==========================================================

import pandas as pd
import re
import json

df = pd.read_csv("banking_charge_cleaned.csv")

states = (
    df["state_clean"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

states = sorted([
    s for s in states
    if s and s.lower() != "nan"
])

state_master = pd.DataFrame({
    "state": states
})

print("="*80)
print("TOTAL STATES:", len(state_master))
print("="*80)

display(state_master)

state_master.to_csv(
    "state_master.csv",
    index=False
)

with open(
    "state_master.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        states,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nSaved:")
print("state_master.csv")
print("state_master.json")

TOTAL STATES: 36


,state
0,*West Bengal
1,Andaman Nicobar Islands
2,Andhra Pradesh
3,Arunachal Pradesh
4,Assam
5,Bihar
6,Chandigarh
7,Chhattisgarh
8,Dadra Nagar Haveli and Daman Diu
9,Delhi



Saved:
state_master.csv
state_master.json


In [ ]:
# ==========================================================
# PHASE 6C
# LOAD STATE MASTER
# ==========================================================

import pandas as pd

state_master = pd.read_csv(
    "state_master.csv"
)

KNOWN_STATES = set(
    state_master["state"]
    .astype(str)
    .str.strip()
)

print(
    "Loaded",
    len(KNOWN_STATES),
    "states"
)

for s in sorted(list(KNOWN_STATES))[:20]:
    print(s)

Loaded 36 states
*West Bengal
Andaman Nicobar Islands
Andhra Pradesh
Arunachal Pradesh
Assam
Bihar
Chandigarh
Chhattisgarh
Dadra Nagar Haveli and Daman Diu
Delhi
Goa
Gujarat
Haryana
Himachal Pradesh
J&K
Jharkhand
Karnataka
Kerala
Ladakh
Lakshadweep


In [ ]:
# ==========================================================
# TEST STATE DETECTION
# ==========================================================

for i in range(25):

    raw = clean_text(data.iloc[i,0])

    name = extract_english(raw)

    is_state = (
        name in KNOWN_STATES
    )

    print(
        f"{i:02d}",
        "|",
        name[:40],
        "|",
        "STATE" if is_state else "UTILITY"
    )

00 | Andhra Pradesh* | UTILITY
01 | APTRANSCO | UTILITY
02 | Arunachal Pradesh | STATE
03 | Arunachal PD | UTILITY
04 | Assam* | UTILITY
05 | AEGCL | UTILITY
06 | Bihar | STATE
07 | BSPTCL | UTILITY
08 | BGCL | UTILITY
09 | Chhattisgarh* | UTILITY
10 | CSPTCL | UTILITY
11 | दादरा और नगर हवेली और दमन और दीव Dadra & | UTILITY
12 | DNHDDPCL | UTILITY
13 | Delhi | STATE
14 | DTL | UTILITY
15 | Gujarat* | UTILITY
16 | GETCO | UTILITY
17 | ह रयाणा Haryana | UTILITY
18 |  | UTILITY
19 | Himachal Pradesh | STATE
20 | HPPTCL | UTILITY
21 | ज ू और क ीर और ल ाख J&K and Ladakh | UTILITY
22 | JKPTCL | UTILITY
23 | Jharkhand | STATE
24 | JUSNL | UTILITY


In [ ]:
# ==========================================================
# PHASE 6E
# CANONICAL STATE NORMALIZER
# ==========================================================

import re

def normalize_state_name(text):

    text = str(text)

    # remove CID garbage remnants
    text = re.sub(r"\(cid:\d+\)", " ", text)

    # keep English side if bilingual
    if "/" in text:
        text = text.split("/")[-1]

    text = text.strip()

    # remove *
    text = text.replace("*", "")

    # normalize &
    text = text.replace("&", "and")

    # collapse spaces
    text = re.sub(r"\s+", " ", text)

    # custom mappings
    replacements = {

        "JandK and Ladakh":
            "J&K and Ladakh",

        "Dadra and Nagar Haveli and Daman and Diu":
            "Dadra Nagar Haveli and Daman Diu",

        "Orissa":
            "Odisha",

        "West Bengal":
            "West Bengal",

        "Nagaland":
            "Nagaland"
    }

    text = replacements.get(text, text)

    return text.strip()


# ----------------------------------------------------------
# TEST
# ----------------------------------------------------------

for i in range(25):

    raw = clean_text(data.iloc[i,0])

    name = extract_english(raw)

    norm = normalize_state_name(name)

    print(
        f"{i:02d}",
        "|",
        norm
    )

00 | Andhra Pradesh
01 | APTRANSCO
02 | Arunachal Pradesh
03 | Arunachal PD
04 | Assam
05 | AEGCL
06 | Bihar
07 | BSPTCL
08 | BGCL
09 | Chhattisgarh
10 | CSPTCL
11 | दादरा और नगर हवेली और दमन और दीव Dadra and Nagar Haveli and Daman and Diu
12 | DNHDDPCL
13 | Delhi
14 | DTL
15 | Gujarat
16 | GETCO
17 | ह रयाणा Haryana
18 | 
19 | Himachal Pradesh
20 | HPPTCL
21 | ज ू और क ीर और ल ाख JandK and Ladakh
22 | JKPTCL
23 | Jharkhand
24 | JUSNL


In [ ]:
canonical_states = set()

for s in KNOWN_STATES:

    canonical_states.add(
        normalize_state_name(s)
    )

print(
    "Canonical States:",
    len(canonical_states)
)

Canonical States: 36


In [ ]:
for i in range(25):

    raw = clean_text(data.iloc[i,0])

    name = extract_english(raw)

    norm = normalize_state_name(name)

    is_state = norm in canonical_states

    print(
        f"{i:02d}",
        "|",
        norm[:40],
        "|",
        "STATE" if is_state else "UTILITY"
    )

00 | Andhra Pradesh | STATE
01 | APTRANSCO | UTILITY
02 | Arunachal Pradesh | STATE
03 | Arunachal PD | UTILITY
04 | Assam | STATE
05 | AEGCL | UTILITY
06 | Bihar | STATE
07 | BSPTCL | UTILITY
08 | BGCL | UTILITY
09 | Chhattisgarh | STATE
10 | CSPTCL | UTILITY
11 | दादरा और नगर हवेली और दमन और दीव Dadra a | UTILITY
12 | DNHDDPCL | UTILITY
13 | Delhi | STATE
14 | DTL | UTILITY
15 | Gujarat | STATE
16 | GETCO | UTILITY
17 | ह रयाणा Haryana | UTILITY
18 |  | UTILITY
19 | Himachal Pradesh | STATE
20 | HPPTCL | UTILITY
21 | ज ू और क ीर और ल ाख JandK and Ladakh | UTILITY
22 | JKPTCL | UTILITY
23 | Jharkhand | STATE
24 | JUSNL | UTILITY


In [ ]:
def extract_english(text):

    text = clean_text(text)

    # keep English letters, numbers, &, -, spaces only
    matches = re.findall(
        r"[A-Za-z&\-]+",
        text
    )

    return " ".join(matches).strip()

In [ ]:
tests = [
    "ह रयाणा Haryana",
    "ज ू और क ीर और ल ाख JandK and Ladakh",
    "दादरा और नगर हवेली और दमन और दीव Dadra and Nagar Haveli and Daman and Diu"
]

for t in tests:
    print(extract_english(t))

Haryana
JandK and Ladakh
Dadra and Nagar Haveli and Daman and Diu


In [ ]:
replacements = {

    "JandK and Ladakh":
        "J&K and Ladakh",

    "Dadra and Nagar Haveli and Daman and Diu":
        "Dadra Nagar Haveli and Daman Diu",

    "Haryana":
        "Haryana"
}

In [ ]:
for i in range(25):

    raw = clean_text(data.iloc[i,0])

    name = extract_english(raw)

    norm = normalize_state_name(name)

    is_state = norm in canonical_states

    print(
        f"{i:02d}",
        "|",
        norm,
        "|",
        "STATE" if is_state else "UTILITY"
    )

00 | Andhra Pradesh | STATE
01 | APTRANSCO | UTILITY
02 | Arunachal Pradesh | STATE
03 | Arunachal PD | UTILITY
04 | Assam | STATE
05 | AEGCL | UTILITY
06 | Bihar | STATE
07 | BSPTCL | UTILITY
08 | BGCL | UTILITY
09 | Chhattisgarh | STATE
10 | CSPTCL | UTILITY
11 | Dadra Nagar Haveli and Daman Diu | STATE
12 | DNHDDPCL | UTILITY
13 | Delhi | STATE
14 | DTL | UTILITY
15 | Gujarat | STATE
16 | GETCO | UTILITY
17 | Haryana | STATE
18 |  | UTILITY
19 | Himachal Pradesh | STATE
20 | HPPTCL | UTILITY
21 | J&K and Ladakh | UTILITY
22 | JKPTCL | UTILITY
23 | Jharkhand | STATE
24 | JUSNL | UTILITY


In [ ]:
# ==========================================================
# PARAMETER REGISTRY
# ==========================================================

PARAMETER_CONFIG = {

    "BANKING CHARGES": {
        "parser": "narrative",
        "start_page": 64,
        "end_page": 75
    },

    "TRANSMISSION CHARGE": {
        "parser": "numeric",
        "start_page": 62,
        "end_page": 63
    },

    "WHEELING CHARGE": {
        "parser": "numeric",
        "start_page": 59,
        "end_page": 60
    },

    "ADDITIONAL SURCHARGE": {
        "parser": "numeric",
        "start_page": 57,
        "end_page": 58
    },

    "CROSS SUBSIDY SURCHARGE": {
        "parser": "numeric",
        "start_page": 50,
        "end_page": 56
    }
}

In [ ]:
def extract_raw_table(
    pdf_path,
    start_page,
    end_page
):

    import pdfplumber
    import pandas as pd

    rows = []

    with pdfplumber.open(pdf_path) as pdf:

        for page_num in range(
            start_page,
            end_page + 1
        ):

            page = pdf.pages[page_num - 1]

            tables = page.extract_tables()

            if not tables:
                continue

            largest = max(
                tables,
                key=lambda t:
                    len(t) *
                    max(len(r) for r in t if r)
            )

            rows.extend(largest)

    return pd.DataFrame(rows)

In [ ]:
def parse_numeric_matrix(df):

    import re
    import pandas as pd

    # -----------------------------------------
    # Remove header rows
    # -----------------------------------------

    df = df.iloc[5:].reset_index(drop=True)

    NAME_COL = 0
    YEAR_COL = 3

    LONG_COL = 6
    LONG_UNIT_COL = 9

    SHORT_COL = 12
    SHORT_UNIT_COL = 15

    records = []

    current_state = None
    current_payload = None

    for _, row in df.iterrows():

        name = extract_english(
            clean_text(row[NAME_COL])
        )

        if not name:
            continue

        norm_name = normalize_state_name(name)

        is_state = (
            canonicalize_state(norm_name)
            in canonical_states
        )

        year = clean_text(row[YEAR_COL])

        long_charge = clean_text(row[LONG_COL])
        long_unit = clean_text(row[LONG_UNIT_COL])

        short_charge = clean_text(row[SHORT_COL])
        short_unit = clean_text(row[SHORT_UNIT_COL])

        # -------------------------------------
        # Parent row
        # -------------------------------------

        if is_state:

            current_state = norm_name

            current_payload = {
                "year": year,
                "long_medium_charge": long_charge,
                "long_medium_unit": long_unit,
                "short_term_charge": short_charge,
                "short_term_unit": short_unit
            }

        # -------------------------------------
        # Child utility row
        # -------------------------------------

        else:

            if current_state and current_payload:

                records.append({
                    "state": current_state,
                    "utility": norm_name,
                    **current_payload
                })

    return records

In [ ]:
def parse_narrative_matrix(df):

    import pandas as pd

    df = df.iloc[4:].reset_index(drop=True)

    current_state = None
    current_master = None

    records = []

    for _, row in df.iterrows():

        state = extract_state_name(
            row.iloc[0]
        )

        discom = clean_text(
            row.iloc[1]
        )

        charge = clean_text(
            row.iloc[2]
        )

        period = clean_text(
            row.iloc[3]
        )

        policy = ""

        if len(row) > 6:
            policy = clean_text(
                row.iloc[6]
            )

        # ---------------------------------
        # State propagation
        # ---------------------------------

        if state:
            current_state = state

        state = current_state

        # ---------------------------------
        # Master row
        # ---------------------------------

        if discom and charge:

            current_master = {
                "state": state,
                "charge": charge,
                "period": period,
                "policy": policy
            }

            records.append({
                "state": state,
                "discom": discom,
                "charge": charge,
                "period": period,
                "policy": policy
            })

        # ---------------------------------
        # Child DISCOM
        # ---------------------------------

        elif discom and not charge:

            if current_master:

                records.append({
                    "state": state,
                    "discom": discom,
                    "charge":
                        current_master["charge"],
                    "period":
                        current_master["period"],
                    "policy":
                        current_master["policy"]
                })

    return records

In [ ]:
STATE_ALIASES = {
    "J&K and Ladakh": "J&K",
    "JandK and Ladakh": "J&K",
    "Orissa": "Odisha",
    "*West Bengal": "West Bengal",
    "s Nagaland": "Nagaland"
}

def canonicalize_state(name):

    name = normalize_state_name(name)

    return STATE_ALIASES.get(name, name)

In [ ]:
def extract_parameter(
    pdf_path,
    parameter_name
):

    cfg = PARAMETER_CONFIG[
        parameter_name
    ]

    raw_df = extract_raw_table(
        pdf_path,
        cfg["start_page"],
        cfg["end_page"]
    )

    parser_type = cfg["parser"]

    if parser_type == "narrative":

        return parse_narrative_matrix(
            raw_df
        )

    elif parser_type == "numeric":

        return parse_numeric_matrix(
            raw_df
        )

    else:

        raise ValueError(
            f"Unknown parser: {parser_type}"
        )

In [ ]:
result = extract_parameter(
    PDF_PATH,
    "TRANSMISSION CHARGE"
)

print(
    "records:",
    len(result)
)

for r in result[:5]:
    print(r)

records: 27
{'state': 'Andhra Pradesh', 'utility': 'APTRANSCO', 'year': '2026-27', 'long_medium_charge': '201.80', 'long_medium_unit': 'Rs/kVA/month', 'short_term_charge': '201.80', 'short_term_unit': 'Rs/kVA/month'}
{'state': 'Arunachal Pradesh', 'utility': 'Arunachal PD', 'year': '2023-24', 'long_medium_charge': '71,253', 'long_medium_unit': 'Rs/MW/month', 'short_term_charge': '0', 'short_term_unit': 'Rs/kWh'}
{'state': 'Assam', 'utility': 'AEGCL', 'year': '2026-27', 'long_medium_charge': '--', 'long_medium_unit': '--', 'short_term_charge': '0.45', 'short_term_unit': 'Rs/kWh'}
{'state': 'Bihar', 'utility': 'BSPTCL', 'year': '2026-27', 'long_medium_charge': '--', 'long_medium_unit': '--', 'short_term_charge': '--', 'short_term_unit': '--'}
{'state': 'Bihar', 'utility': 'BGCL', 'year': '2026-27', 'long_medium_charge': '--', 'long_medium_unit': '--', 'short_term_charge': '--', 'short_term_unit': '--'}


In [ ]:
result = extract_parameter(
    PDF_PATH,
    "BANKING CHARGES"
)

print("records:", len(result))
print(result[:5])

records: 66
[{'state': 'Andaman Nicobar Islands', 'discom': 'ED-A&NI', 'charge': '8.00%', 'period': 'Monthly/ Billing Cycle', 'policy': "1. नवीकरणीय ऊजा उ ादन क ारा िकए गए कुल उ ादन के 20% तक सीिमत: शेष बची िवद्युत, की 'डी ड खरीद' मानी जाएगी, िजसका भुगतान औसत िवद्यु रीद लागत (एपीपीसी) या उस वष के िलए िनधा रत फीड-इन-टै रफ (िबना स डी और रत मू ास के लाभ को जोड़े), दोनो ंम से जो भी कम हो, उस दर पर िकया जाएगा। Limited to 30% of the total generation by RE Generating Station: Deemed purchase at Average Power Purchase Cost (APPC) or Feed-in-Tariff determined for that year without considering subsidy and Accelerated Depreciation, whichever is lower. 2. नवीकरणीय ऊजा उ ादन क ारा िकए गए कुल उ ादन के 20% से अिधक होने पर: यह ऊजा तः समा हो जाएगी और इसके िलए कोई मुआवजा नही ंिदया जाएगा। In excess of 20% of the total generation by RE Generating Station: To lapse and no compensation."}, {'state': 'Andhra Pradesh', 'discom': 'APEPDCL', 'charge': '8.00%', 'period': 'Monthly/ Billing Cycle', 'policy': 'ीन एन

In [ ]:
banking = extract_parameter(
    PDF_PATH,
    "BANKING CHARGES"
)

print("records:", len(banking))

states = {}

for r in banking:

    s = r["state"]

    states[s] = states.get(s, 0) + 1

print("\nState Counts\n")

for k,v in sorted(states.items()):
    print(k, ":", v)

records: 66

State Counts

Andaman Nicobar Islands : 1
Andhra Pradesh : 3
Arunachal Pradesh : 1
Assam : 1
Bihar : 2
Chandigarh : 1
Chhattisgarh : 1
Dadra Nagar Haveli and Daman Diu : 1
Delhi : 3
Goa : 1
Gujarat : 4
Haryana : 2
Himachal Pradesh : 1
J&K : 2
Jharkhand : 1
Karnataka : 4
Kerala : 1
Ladakh : 1
Lakshadweep : 1
Madhya Pradesh : 3
Maharashtra : 1
Manipur : 1
Meghalaya : 1
Mizoram : 1
Odisha : 1
Puducherry : 1
Punjab : 1
Rajasthan : 1
Sikkim : 1
State : 14
Tamil Nadu : 1
Telangana : 2
Tripura : 1
Uttar Pradesh : 1
Uttarakhand : 1
West Bengal : 1
sनागाल Nagaland : 1


In [ ]:
# ==========================================================
# PHASE 7A
# PRODUCTION BANKING PARSER V2
# HEADER REMOVAL + OCR CLEANUP
# ==========================================================

import pandas as pd
import re

# ----------------------------------------------------------
# HEADER DETECTOR
# ----------------------------------------------------------

HEADER_PATTERNS = [

    "state",
    "discom",
    "description of banking",
    "banking charge",
    "banking charges",
    "period of banking",
    "banking period",
    "description",
    "utility",
    "open access charges"
]

def is_header_row(row):

    text = " ".join(
        [
            str(x)
            for x in row
            if pd.notna(x)
        ]
    ).lower()

    text = re.sub(r"\s+", " ", text)

    matches = 0

    for p in HEADER_PATTERNS:

        if p in text:
            matches += 1

    return matches >= 2


# ----------------------------------------------------------
# OCR CLEANUP
# ----------------------------------------------------------

def clean_state_text(text):

    text = clean_text(text)

    text = re.sub(r"[^A-Za-z&\-\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    replacements = {

        "s Nagaland":
            "Nagaland",

        "Nagaland":
            "Nagaland",

        "JandK and Ladakh":
            "J&K and Ladakh",

        "Dadra and Nagar Haveli and Daman and Diu":
            "Dadra Nagar Haveli and Daman Diu"
    }

    return replacements.get(text, text)


# ----------------------------------------------------------
# BANKING PARSER
# ----------------------------------------------------------

def parse_narrative_matrix(df):

    records = []

    current_state = None
    current_master = None

    for _, row in df.iterrows():

        # ---------------------------------------------
        # REMOVE REPEATED PAGE HEADERS
        # ---------------------------------------------

        if is_header_row(row):
            continue

        # ---------------------------------------------
        # EXTRACT FIELDS
        # ---------------------------------------------

        state = clean_state_text(
            extract_english(
                row.iloc[0]
            )
        )

        discom = clean_text(
            row.iloc[1]
        )

        charge = clean_text(
            row.iloc[2]
        )

        period = clean_text(
            row.iloc[3]
        )

        policy = ""

        if len(row) > 6:

            policy = clean_text(
                row.iloc[6]
            )

        # ---------------------------------------------
        # IGNORE EMPTY ROWS
        # ---------------------------------------------

        if (
            not state
            and not discom
            and not charge
            and not period
            and not policy
        ):
            continue

        # ---------------------------------------------
        # IGNORE HEADER GARBAGE
        # ---------------------------------------------

        if state.lower() == "state":
            continue

        if discom.lower() == "discom":
            continue

        # ---------------------------------------------
        # STATE PROPAGATION
        # ---------------------------------------------

        if state:
            current_state = canonicalize_state(
                state
            )

        state = current_state

        # ---------------------------------------------
        # MASTER RECORD
        # ---------------------------------------------

        if discom and charge:

            current_master = {

                "state": state,
                "charge": charge,
                "period": period,
                "policy": policy
            }

            records.append({

                "state": state,
                "discom": discom,
                "charge": charge,
                "period": period,
                "policy": policy
            })

        # ---------------------------------------------
        # CHILD DISCOM
        # ---------------------------------------------

        elif discom and not charge:

            if current_master:

                records.append({

                    "state": state,
                    "discom": discom,
                    "charge":
                        current_master["charge"],

                    "period":
                        current_master["period"],

                    "policy":
                        current_master["policy"]
                })

        # ---------------------------------------------
        # CONTINUATION POLICY
        # ---------------------------------------------

        elif policy:

            if records:

                records[-1]["policy"] += (
                    "\n" + policy
                )

    return records


# ==========================================================
# TEST
# ==========================================================

banking = extract_parameter(
    PDF_PATH,
    "BANKING CHARGES"
)

print("records:", len(banking))

states = {}

for r in banking:

    s = r["state"]

    states[s] = states.get(s, 0) + 1

print("\nState Counts\n")

for k,v in sorted(states.items()):
    print(k, ":", v)

records: 53

State Counts

Andaman and Nicobar Islands : 1
Andhra Pradesh : 3
Arunachal Pradesh : 1
Assam : 1
Bihar : 2
Chandigarh : 1
Chhattisgarh : 1
Dadra Nagar Haveli and Daman Diu : 1
Delhi : 3
Goa : 1
Gujarat : 4
Haryana : 2
Himachal Pradesh : 1
JandK : 2
Jharkhand : 1
Karnataka : 5
Kerala : 1
Ladakh : 1
Lakshadweep : 1
Madhya Pradesh : 3
Maharashtra : 1
Manipur : 1
Meghalaya : 1
Mizoram : 1
Nagaland : 1
Odisha : 1
Puducherry : 1
Punjab : 1
Rajasthan : 3
Sikkim : 1
Tamil Nadu : 2
Tripura : 1
Uttarakhand : 1
West Bengal : 1


In [ ]:
# ==========================================================
# PHASE 7B
# TRANSMISSION AUDIT
# VERIFY PARENT -> CHILD INHERITANCE
# ==========================================================

import pandas as pd
import json

# ----------------------------------------------------------
# LOAD TRANSMISSION OUTPUT
# ----------------------------------------------------------

transmission = extract_parameter(
    PDF_PATH,
    "TRANSMISSION CHARGE"
)

print("=" * 80)
print("TOTAL RECORDS:", len(transmission))
print("=" * 80)

# ----------------------------------------------------------
# STATE COUNTS
# ----------------------------------------------------------

state_counts = {}

for r in transmission:

    state = r["state"]

    state_counts[state] = (
        state_counts.get(state, 0) + 1
    )

print("\nSTATE COUNTS\n")

for k,v in sorted(state_counts.items()):
    print(f"{k} : {v}")

# ----------------------------------------------------------
# LOOK FOR BAD STATES
# ----------------------------------------------------------

print("\n")
print("=" * 80)
print("CHECKING FOR INVALID STATE NAMES")
print("=" * 80)

bad_states = []

for s in sorted(state_counts):

    if len(s.strip()) < 3:

        bad_states.append(s)

    if "STATE" in s.upper():

        bad_states.append(s)

    if "UTILITY" in s.upper():

        bad_states.append(s)

print("BAD STATES:", bad_states)

# ----------------------------------------------------------
# LOOK FOR STATE USED AS UTILITY
# ----------------------------------------------------------

print("\n")
print("=" * 80)
print("CHECKING STATE->UTILITY LEAKAGE")
print("=" * 80)

state_names = set(
    state_counts.keys()
)

leaks = []

for r in transmission:

    utility = r["utility"]

    if utility in state_names:

        leaks.append(r)

print("LEAK COUNT:", len(leaks))

for r in leaks[:20]:

    print(
        json.dumps(
            r,
            indent=2,
            ensure_ascii=False
        )
    )

# ----------------------------------------------------------
# SAMPLE RECORDS
# ----------------------------------------------------------

print("\n")
print("=" * 80)
print("FIRST 30 RECORDS")
print("=" * 80)

for r in transmission[:30]:

    print(
        json.dumps(
            r,
            indent=2,
            ensure_ascii=False
        )
    )
    print("-" * 50)

# ----------------------------------------------------------
# CHECK EMPTY VALUES
# ----------------------------------------------------------

print("\n")
print("=" * 80)
print("EMPTY VALUE AUDIT")
print("=" * 80)

empty_year = 0
empty_long = 0
empty_short = 0

for r in transmission:

    if not r["year"]:
        empty_year += 1

    if not r["long_medium_charge"]:
        empty_long += 1

    if not r["short_term_charge"]:
        empty_short += 1

print("EMPTY YEAR:", empty_year)
print("EMPTY LONG CHARGE:", empty_long)
print("EMPTY SHORT CHARGE:", empty_short)

# ----------------------------------------------------------
# BIHAR VALIDATION
# ----------------------------------------------------------

print("\n")
print("=" * 80)
print("BIHAR RECORDS")
print("=" * 80)

for r in transmission:

    if r["state"] == "Bihar":

        print(
            json.dumps(
                r,
                indent=2,
                ensure_ascii=False
            )
        )
        print("-" * 50)

TOTAL RECORDS: 27

STATE COUNTS

Andhra Pradesh : 1
Arunachal Pradesh : 1
Assam : 1
Bihar : 2
Chhattisgarh : 1
Dadra Nagar Haveli and Daman Diu : 1
Delhi : 1
Gujarat : 1
Himachal Pradesh : 3
Jharkhand : 1
Karnataka : 1
Kerala : 1
Madhya Pradesh : 1
Maharashtra : 1
Manipur : 1
Meghalaya : 1
Odisha : 1
Punjab : 1
Rajasthan : 1
Tamil Nadu : 1
Telangana : 1
Tripura : 1
Uttar Pradesh : 1
Uttarakhand : 1


CHECKING FOR INVALID STATE NAMES
BAD STATES: []


CHECKING STATE->UTILITY LEAKAGE
LEAK COUNT: 0


FIRST 30 RECORDS
{
  "state": "Andhra Pradesh",
  "utility": "APTRANSCO",
  "year": "2026-27",
  "long_medium_charge": "201.80",
  "long_medium_unit": "Rs/kVA/month",
  "short_term_charge": "201.80",
  "short_term_unit": "Rs/kVA/month"
}
--------------------------------------------------
{
  "state": "Arunachal Pradesh",
  "utility": "Arunachal PD",
  "year": "2023-24",
  "long_medium_charge": "71,253",
  "long_medium_unit": "Rs/MW/month",
  "short_term_charge": "0",
  "short_term_unit": "Rs/k

In [ ]:
# ==========================================================
# PHASE 8A
# UNIVERSAL NUMERIC PARSER V2
# STATE MASTER DRIVEN
# HEADER REMOVAL
# PARENT ROW DETECTION
# ==========================================================

import pandas as pd
import re

# ----------------------------------------------------------
# NUMERIC HEADER DETECTOR
# ----------------------------------------------------------

NUMERIC_HEADER_PATTERNS = [

    "state",
    "utility",
    "transmission charge",
    "wheeling charge",
    "additional surcharge",
    "cross subsidy surcharge",
    "states uts",
    "financial year",
    "long term",
    "medium term",
    "short term",
    "unit"
]

def is_numeric_header_row(row):

    text = " ".join(
        [
            str(x)
            for x in row
            if pd.notna(x)
        ]
    ).lower()

    matches = 0

    for p in NUMERIC_HEADER_PATTERNS:

        if p in text:
            matches += 1

    return matches >= 2


# ----------------------------------------------------------
# NUMERIC PARSER
# ----------------------------------------------------------

def parse_numeric_matrix(df):

    NAME_COL = 0
    YEAR_COL = 3

    LONG_COL = 6
    LONG_UNIT_COL = 9

    SHORT_COL = 12
    SHORT_UNIT_COL = 15

    records = []

    current_state = None
    current_payload = None

    for _, row in df.iterrows():

        # -----------------------------------------
        # REMOVE PAGE HEADERS
        # -----------------------------------------

        if is_numeric_header_row(row):
            continue

        raw_name = clean_text(
            row[NAME_COL]
        )

        if not raw_name:
            continue

        name = normalize_state_name(
            extract_english(raw_name)
        )

        if not name:
            continue

        # -----------------------------------------
        # REMOVE KNOWN GARBAGE
        # -----------------------------------------

        if name in [
            "States UTs",
            "State",
            "Utility"
        ]:
            continue

        # -----------------------------------------
        # STATE DETECTION
        # -----------------------------------------

        canonical_name = canonicalize_state(
            name
        )

        is_state = (
            canonical_name
            in canonical_states
        )

        year = clean_text(
            row[YEAR_COL]
        )

        long_charge = clean_text(
            row[LONG_COL]
        )

        long_unit = clean_text(
            row[LONG_UNIT_COL]
        )

        short_charge = clean_text(
            row[SHORT_COL]
        )

        short_unit = clean_text(
            row[SHORT_UNIT_COL]
        )

        # -----------------------------------------
        # PARENT STATE ROW
        # -----------------------------------------

        if is_state:

            current_state = canonical_name

            current_payload = {

                "year": year,

                "long_medium_charge":
                    long_charge,

                "long_medium_unit":
                    long_unit,

                "short_term_charge":
                    short_charge,

                "short_term_unit":
                    short_unit
            }

            continue

        # -----------------------------------------
        # SAFETY CHECK
        # -----------------------------------------

        if current_state is None:
            continue

        if current_payload is None:
            continue

        # -----------------------------------------
        # UTILITY RECORD
        # -----------------------------------------

        records.append({

            "state":
                current_state,

            "utility":
                name,

            **current_payload
        })

    return records


# ==========================================================
# TEST
# ==========================================================

transmission = extract_parameter(
    PDF_PATH,
    "TRANSMISSION CHARGE"
)

print("=" * 80)
print("RECORDS:", len(transmission))
print("=" * 80)

for r in transmission[:20]:

    print(r)

RECORDS: 26
{'state': 'Andhra Pradesh', 'utility': 'APTRANSCO', 'year': '2026-27', 'long_medium_charge': '201.80', 'long_medium_unit': 'Rs/kVA/month', 'short_term_charge': '201.80', 'short_term_unit': 'Rs/kVA/month'}
{'state': 'Arunachal Pradesh', 'utility': 'Arunachal PD', 'year': '2023-24', 'long_medium_charge': '71,253', 'long_medium_unit': 'Rs/MW/month', 'short_term_charge': '0', 'short_term_unit': 'Rs/kWh'}
{'state': 'Assam', 'utility': 'AEGCL', 'year': '2026-27', 'long_medium_charge': '--', 'long_medium_unit': '--', 'short_term_charge': '0.45', 'short_term_unit': 'Rs/kWh'}
{'state': 'Bihar', 'utility': 'BSPTCL', 'year': '2026-27', 'long_medium_charge': '--', 'long_medium_unit': '--', 'short_term_charge': '--', 'short_term_unit': '--'}
{'state': 'Bihar', 'utility': 'BGCL', 'year': '2026-27', 'long_medium_charge': '--', 'long_medium_unit': '--', 'short_term_charge': '--', 'short_term_unit': '--'}
{'state': 'Chhattisgarh', 'utility': 'CSPTCL', 'year': '2025-26', 'long_medium_charge'

In [ ]:
# ==========================================================
# PHASE 8B
# REBUILD CANONICAL STATE MASTER
# ==========================================================

canonical_states = {

    "Andaman and Nicobar Islands",
    "Andhra Pradesh",
    "Arunachal Pradesh",
    "Assam",
    "Bihar",
    "Chandigarh",
    "Chhattisgarh",
    "Dadra Nagar Haveli and Daman Diu",
    "Delhi",
    "Goa",
    "Gujarat",
    "Haryana",
    "Himachal Pradesh",
    "J&K and Ladakh",
    "J&K",
    "Ladakh",
    "Jharkhand",
    "Karnataka",
    "Kerala",
    "Lakshadweep",
    "Madhya Pradesh",
    "Maharashtra",
    "Manipur",
    "Meghalaya",
    "Mizoram",
    "Nagaland",
    "Odisha",
    "Puducherry",
    "Punjab",
    "Rajasthan",
    "Sikkim",
    "Tamil Nadu",
    "Telangana",
    "Tripura",
    "Uttar Pradesh",
    "Uttarakhand",
    "West Bengal"
}

print(
    "Canonical states:",
    len(canonical_states)
)

for s in sorted(canonical_states):
    print(s)

Canonical states: 37
Andaman and Nicobar Islands
Andhra Pradesh
Arunachal Pradesh
Assam
Bihar
Chandigarh
Chhattisgarh
Dadra Nagar Haveli and Daman Diu
Delhi
Goa
Gujarat
Haryana
Himachal Pradesh
J&K
J&K and Ladakh
Jharkhand
Karnataka
Kerala
Ladakh
Lakshadweep
Madhya Pradesh
Maharashtra
Manipur
Meghalaya
Mizoram
Nagaland
Odisha
Puducherry
Punjab
Rajasthan
Sikkim
Tamil Nadu
Telangana
Tripura
Uttar Pradesh
Uttarakhand
West Bengal


In [ ]:
# ==========================================================
# PHASE 8C
# TRANSMISSION RE-RUN
# ==========================================================

transmission = extract_parameter(
    PDF_PATH,
    "TRANSMISSION CHARGE"
)

print("="*80)
print("RECORDS:", len(transmission))
print("="*80)

for r in transmission:

    if r["state"] in [
        "Himachal Pradesh",
        "J&K",
        "J&K and Ladakh",
        "Ladakh"
    ]:

        print(r)

RECORDS: 25
{'state': 'Himachal Pradesh', 'utility': 'HPPTCL', 'year': '2026-27', 'long_medium_charge': '--', 'long_medium_unit': '--', 'short_term_charge': '0.10', 'short_term_unit': 'Rs/kWh'}
{'state': 'J&K', 'utility': 'JKPTCL', 'year': '2025-26', 'long_medium_charge': '17.87', 'long_medium_unit': 'Rs. Cr./month', 'short_term_charge': '1,815.55', 'short_term_unit': 'Rs/MW/Day'}


In [ ]:
# ==========================================================
# PHASE 9A
# UNIVERSAL SCHEMA DISCOVERY ENGINE
# ==========================================================

import pdfplumber
import pandas as pd
import re

# ----------------------------------------------------------
# INSPECT ANY PARAMETER TABLE
# ----------------------------------------------------------

def inspect_parameter_schema(
    pdf_path,
    start_page,
    end_page
):

    rows = []

    with pdfplumber.open(pdf_path) as pdf:

        for page_num in range(
            start_page,
            end_page + 1
        ):

            page = pdf.pages[
                page_num - 1
            ]

            tables = page.extract_tables()

            if not tables:
                continue

            largest = max(
                tables,
                key=lambda t:
                    len(t) *
                    max(
                        len(r)
                        for r in t
                        if r
                    )
            )

            rows.extend(largest)

    df = pd.DataFrame(rows)

    print("="*100)
    print("SHAPE:", df.shape)
    print("="*100)

    # --------------------------------------
    # COLUMN DENSITY
    # --------------------------------------

    print("\nCOLUMN DENSITY\n")

    active_cols = []

    for col in df.columns:

        non_null = (
            df[col]
            .fillna("")
            .astype(str)
            .str.strip()
            .replace("", pd.NA)
            .notna()
            .sum()
        )

        if non_null > 0:

            active_cols.append(col)

            print(
                f"COL {col} -> {non_null}"
            )

    print("\nACTIVE COLS:")
    print(active_cols)

    # --------------------------------------
    # SAMPLE DATA
    # --------------------------------------

    print("\n")
    print("="*100)
    print("FIRST 20 ROWS")
    print("="*100)

    pd.set_option(
        "display.max_columns",
        None
    )

    pd.set_option(
        "display.max_colwidth",
        200
    )

    display(
        df.head(20)
    )

    return df

# ==========================================================
# CSS
# ==========================================================

css_df = inspect_parameter_schema(
    PDF_PATH,
    50,
    56
)

# ==========================================================
# ADDITIONAL SURCHARGE
# ==========================================================

additional_df = inspect_parameter_schema(
    PDF_PATH,
    57,
    58
)

# ==========================================================
# WHEELING
# ==========================================================

wheeling_df = inspect_parameter_schema(
    PDF_PATH,
    59,
    60
)

SHAPE: (324, 21)

COLUMN DENSITY

COL 0 -> 248
COL 1 -> 74
COL 2 -> 6
COL 3 -> 202
COL 4 -> 87
COL 5 -> 6
COL 6 -> 70
COL 7 -> 16
COL 8 -> 20
COL 9 -> 54
COL 10 -> 12
COL 11 -> 26
COL 12 -> 25
COL 13 -> 43
COL 14 -> 2
COL 15 -> 23
COL 16 -> 18
COL 17 -> 1
COL 18 -> 21
COL 19 -> 5

ACTIVE COLS:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


FIRST 20 ROWS


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
0,,आं(cid:364) (cid:366)देश / Andhra Pradesh,,,2026-27,None,None,None,None,None,None,,None,None,None,None,None,None,None,None,None
1,,Category,,,APSPDCL,,,APEPDCL,,,APCPDCL,,None,None,None,None,None,None,None,None,None
2,,HT Category at 11 kV,None,None,None,None,None,None,None,None,None,,None,None,None,None,None,None,None,None,None
3,"HT I(B) Townships, Colonies, Gated\nCommunities and Villas",None,None,0.97,None,None,0.89,None,None,1.12,None,None,None,None,None,None,None,None,None,None,None
4,HT II (A) Commercial& Others,None,None,2.09,None,None,2.20,None,None,2.32,None,None,None,None,None,None,None,None,None,None,None
5,HT II (A) Function Halls/ Auditoriums,None,None,1.99,None,None,1.99,None,None,1.99,None,None,None,None,None,None,None,None,None,None,None
6,HT II (B) Start-up power,None,None,1.99,None,None,1.99,None,None,-,None,None,None,None,None,None,None,None,None,None,None
7,HT III(A) Industry,None,None,1.89,None,None,1.71,None,None,1.85,None,None,None,None,None,None,None,None,None,None,None
8,HT III(B) Seasonal Industries (Off Season),None,None,2.38,None,None,1.76,None,None,1.74,None,None,None,None,None,None,None,None,None,None,None
9,HT III (C ) Energy Intensive Industries,None,None,1.58,None,None,-,None,None,1.75,None,None,None,None,None,None,None,None,None,None,None


SHAPE: (53, 9)

COLUMN DENSITY

COL 0 -> 38
COL 1 -> 11
COL 3 -> 38
COL 4 -> 5
COL 6 -> 42
COL 7 -> 5

ACTIVE COLS:
[0, 1, 3, 4, 6, 7]


FIRST 20 ROWS


,0,1,2,3,4,5,6,7,8
0,,रा(cid:475)/के(cid:574) शािसत (cid:366)देश,,,लागू अविध,,,अित(cid:303)र(cid:389) अिधभार,
1,None,States/UTs,None,None,Applicable Period,None,None,Additional Surcharge,None
2,,िन(cid:635) अित(cid:303)र(cid:389) अिधभार (cid:721)र: ≤ 1.50 (cid:348)पये,None,None,None,None,None,None,
3,None,Low Additional Surcharge Level: ≤ Rs 1.50,None,None,None,None,None,None,None
4,आं(cid:364) (cid:366)देश / Andhra Pradesh*,None,None,2026-27,None,None,Nil,None,None
5,चंडीगढ़ / Chandigarh,None,None,2026-27,None,None,1.32,None,None
6,गोवा / Goa,None,None,2026-27,None,None,0.77,None,None
7,गुजरात / Gujarat,None,None,2025-26,None,None,1st Apr’25 – 30th Sep’25: 0.82,None,None
8,ह(cid:303)रयाणा / Haryana,None,None,2025-26,None,None,Apr-Sep’25 - 1.21,None,None
9,,िहमाचल (cid:366)देश / Himachal Pradesh,,,2026-27,,,0.64,


SHAPE: (86, 24)

COLUMN DENSITY

COL 0 -> 44
COL 1 -> 32
COL 2 -> 28
COL 3 -> 18
COL 4 -> 16
COL 5 -> 28
COL 6 -> 20
COL 7 -> 17
COL 8 -> 26
COL 9 -> 16
COL 10 -> 16
COL 11 -> 20
COL 12 -> 8
COL 13 -> 4
COL 14 -> 20
COL 15 -> 11
COL 16 -> 1
COL 17 -> 23
COL 18 -> 44
COL 19 -> 8
COL 21 -> 27
COL 22 -> 8

ACTIVE COLS:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 21, 22]


FIRST 20 ROWS


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
0,,(cid:684)ीिलंग शु(cid:651) / Wheeling Charges,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,,None,None,None
1,रा(cid:475)/क(cid:336)(cid:363) शािसत (cid:366)देश / States/UTs,None,लागू\nअविध\n(िव(cid:517) वष(cid:330))\nApplicable\nPeriod\n(FY),None,None,,वो(cid:656)ेज (cid:721)र / Voltage Level,None,None,None,None,None,None,None,None,None,None,None,None,None,,None,None,None
2,None,None,None,None,None,11 केवी से\nनीचे\nBelow\n11 kV,None,None,11 केवी\n11 kV,None,None,33 केवी\n33 kV,None,None,66 केवी\n66 kV,None,None,132 केवी\n132 kV,,200 केवी,,None,None,None
3,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,और उससे,None,None,None,None
4,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,अिधक,None,None,None,None
5,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,200 kV &,None,None,None,None
6,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Above,None,None,None,None
7,आं(cid:364) (cid:366)देश / Andhra Pradesh,None,2026-27,None,None,-,None,None,-,None,None,-,None,None,-,None,None,-,-,None,None,None,None,None
8,एपीईपीडीसीएल / APEPDCL,None,2026-27,None,None,NA,None,None,1.00,None,None,0.47,None,None,--,None,None,0.28,0.28,None,None,None,None,None
9,एपीईपीडीसीएल/ APSPDCL,None,2026-27,None,None,NA,None,None,1.00,None,None,0.47,None,None,--,None,None,0.28,0.28,None,None,None,None,None


In [ ]:
# ==========================================================
# EXPORT BANKING + TRANSMISSION
# ==========================================================

import pandas as pd

# ----------------------------------------------------------
# DATAFRAMES
# ----------------------------------------------------------

banking_df = pd.DataFrame(banking)

transmission_df = pd.DataFrame(transmission)

print("Banking Records      :", len(banking_df))
print("Transmission Records :", len(transmission_df))

# ----------------------------------------------------------
# EXPORT
# ----------------------------------------------------------

OUTPUT_FILE = "Regulatory_Parameters.xlsx"

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    banking_df.to_excel(
        writer,
        sheet_name="Banking_Charges",
        index=False
    )

    transmission_df.to_excel(
        writer,
        sheet_name="Transmission_Charges",
        index=False
    )

print(f"\nSaved: {OUTPUT_FILE}")

Banking Records      : 53
Transmission Records : 25

Saved: Regulatory_Parameters.xlsx


In [ ]:
# ==========================================================
# FORMAT EXCEL
# ==========================================================

from openpyxl import load_workbook
from openpyxl.styles import Font

wb = load_workbook(
    "Regulatory_Parameters.xlsx"
)

for ws in wb.worksheets:

    # Freeze first row
    ws.freeze_panes = "A2"

    # Bold headers
    for cell in ws[1]:
        cell.font = Font(bold=True)

    # Auto width
    for column in ws.columns:

        max_len = 0

        col_letter = column[0].column_letter

        for cell in column:

            try:
                max_len = max(
                    max_len,
                    len(str(cell.value))
                )
            except:
                pass

        ws.column_dimensions[
            col_letter
        ].width = min(
            max_len + 2,
            60
        )

wb.save(
    "Regulatory_Parameters.xlsx"
)

print("Formatting applied.")
print("Saved: Regulatory_Parameters.xlsx")

Formatting applied.
Saved: Regulatory_Parameters.xlsx


In [ ]:
# ==========================================================
# PHASE 9B
# ADDITIONAL SURCHARGE PARSER
# ==========================================================

import pandas as pd

# ----------------------------------------------------------
# PARSER
# ----------------------------------------------------------

def parse_additional_surcharge(df):

    records = []

    current_state = None

    for _, row in df.iterrows():

        state = clean_text(
            extract_english(
                row.iloc[0]
            )
        )

        year = clean_text(
            row.iloc[3]
        )

        surcharge = clean_text(
            row.iloc[6]
        )

        # skip headers

        row_text = " ".join(
            [
                str(x)
                for x in row
                if pd.notna(x)
            ]
        ).lower()

        if (
            "additional surcharge" in row_text
            or "states/uts" in row_text
            or "applicable period" in row_text
        ):
            continue

        if state:

            state = canonicalize_state(
                state
            )

            current_state = state

        else:

            state = current_state

        if not state:
            continue

        if not year:
            continue

        if not surcharge:
            continue

        records.append({

            "state":
                state,

            "year":
                year,

            "additional_surcharge":
                surcharge
        })

    return records


# ----------------------------------------------------------
# EXTRACTION
# ----------------------------------------------------------

def extract_additional_surcharge():

    rows = []

    with pdfplumber.open(PDF_PATH) as pdf:

        for page_num in [57, 58]:

            page = pdf.pages[
                page_num - 1
            ]

            tables = page.extract_tables()

            if not tables:
                continue

            largest = max(
                tables,
                key=lambda t:
                    len(t) *
                    max(
                        len(r)
                        for r in t
                        if r
                    )
            )

            rows.extend(
                largest
            )

    df = pd.DataFrame(rows)

    return parse_additional_surcharge(df)


# ----------------------------------------------------------
# TEST
# ----------------------------------------------------------

additional = extract_additional_surcharge()

print(
    "Records:",
    len(additional)
)

for r in additional[:20]:

    print(r)

Records: 37
{'state': 'Andhra Pradesh', 'year': '2026-27', 'additional_surcharge': 'Nil'}
{'state': 'Chandigarh', 'year': '2026-27', 'additional_surcharge': '1.32'}
{'state': 'Goa', 'year': '2026-27', 'additional_surcharge': '0.77'}
{'state': 'Gujarat', 'year': '2025-26', 'additional_surcharge': '1st Apr’25 – 30th Sep’25: 0.82'}
{'state': 'Haryana', 'year': '2025-26', 'additional_surcharge': 'Apr-Sep’25 - 1.21'}
{'state': 'Karnataka', 'year': '2026-27', 'additional_surcharge': 'Nil'}
{'state': 'Kerala', 'year': '2026-27', 'additional_surcharge': 'Nil'}
{'state': 'Madhya Pradesh', 'year': '2026-27', 'additional_surcharge': '1.18'}
{'state': 'Maharashtra', 'year': '2026-27', 'additional_surcharge': 'Nil'}
{'state': 'Odisha', 'year': '2026-27', 'additional_surcharge': 'Nil'}
{'state': 'Punjab', 'year': '2025-26', 'additional_surcharge': 'Apr-Sep’25 – 1.13 (Partial OA) 1.53 (Full OA)'}
{'state': 'Puducherry', 'year': '2026-27', 'additional_surcharge': '1.43'}
{'state': 'Rajasthan', 'year':

In [ ]:
# ==========================================================
# PHASE 9C
# ADDITIONAL SURCHARGE V2
# STATE + UTILITY SUPPORT
# ==========================================================

import pandas as pd

def parse_additional_surcharge(df):

    records = []

    current_state = None

    for _, row in df.iterrows():

        state_raw = clean_text(
            extract_english(
                row.iloc[0]
            )
        )

        year = clean_text(
            row.iloc[3]
        )

        surcharge = clean_text(
            row.iloc[6]
        )

        row_text = " ".join(
            [
                str(x)
                for x in row
                if pd.notna(x)
            ]
        ).lower()

        # -------------------------------------
        # HEADER REMOVAL
        # -------------------------------------

        if (
            "additional surcharge" in row_text
            or "states/uts" in row_text
            or "applicable period" in row_text
            or "low additional surcharge" in row_text
            or "high additional surcharge" in row_text
        ):
            continue

        if not state_raw:
            continue

        normalized = canonicalize_state(
            state_raw
        )

        is_state = (
            normalized
            in canonical_states
        )

        # -------------------------------------
        # STATE ROW
        # -------------------------------------

        if is_state:

            current_state = normalized

            records.append({

                "state":
                    normalized,

                "utility":
                    None,

                "year":
                    year,

                "additional_surcharge":
                    surcharge
            })

        # -------------------------------------
        # UTILITY ROW
        # -------------------------------------

        else:

            if current_state is None:
                continue

            records.append({

                "state":
                    current_state,

                "utility":
                    state_raw,

                "year":
                    year,

                "additional_surcharge":
                    surcharge
            })

    return records


# ==========================================================
# RE-RUN
# ==========================================================

additional = extract_additional_surcharge()

print("Records:", len(additional))

print("\nDELHI SECTION\n")

for r in additional:

    if r["state"] == "Delhi":

        print(r)

Records: 37

DELHI SECTION

{'state': 'Delhi', 'utility': None, 'year': '2021-22', 'additional_surcharge': '(Oct-Apr): 1.33-1.90'}
{'state': 'Delhi', 'utility': 'BRPL', 'year': '2021-22', 'additional_surcharge': '(Oct-Apr): 1.70'}
{'state': 'Delhi', 'utility': 'BYPL', 'year': '2021-22', 'additional_surcharge': '(Oct-Apr): 1.33'}
{'state': 'Delhi', 'utility': 'TPDDL', 'year': '2021-22', 'additional_surcharge': '(Oct-Apr): 1.90'}


In [ ]:
# ==========================================================
# PHASE 10A
# WHEELING SCHEMA DISCOVERY
# ==========================================================

import pdfplumber
import pandas as pd

rows = []

with pdfplumber.open(PDF_PATH) as pdf:

    for page_num in [59, 60]:

        page = pdf.pages[page_num - 1]

        tables = page.extract_tables()

        if not tables:
            continue

        largest = max(
            tables,
            key=lambda t:
                len(t) *
                max(
                    len(r)
                    for r in t
                    if r
                )
        )

        rows.extend(largest)

wheeling_df = pd.DataFrame(rows)

print("="*100)
print("SHAPE:", wheeling_df.shape)
print("="*100)

for i in range(12):

    print("\nROW", i)
    print("-"*50)

    row = wheeling_df.iloc[i]

    for col, val in enumerate(row):

        if pd.notna(val) and str(val).strip():

            print(
                f"COL {col}:",
                repr(str(val))
            )

SHAPE: (86, 24)

ROW 0
--------------------------------------------------
COL 1: '(cid:684)ीिलंग शु(cid:651) / Wheeling Charges'

ROW 1
--------------------------------------------------
COL 0: 'रा(cid:475)/क(cid:336)(cid:363) शािसत (cid:366)देश / States/UTs'
COL 2: 'लागू\nअविध\n(िव(cid:517) वष(cid:330))\nApplicable\nPeriod\n(FY)'
COL 6: 'वो(cid:656)ेज (cid:721)र / Voltage Level'

ROW 2
--------------------------------------------------
COL 5: '11 केवी से\nनीचे\nBelow\n11 kV'
COL 8: '11 केवी\n11 kV'
COL 11: '33 केवी\n33 kV'
COL 14: '66 केवी\n66 kV'
COL 17: '132 केवी\n132 kV'
COL 19: '200 केवी'

ROW 3
--------------------------------------------------
COL 19: 'और उससे'

ROW 4
--------------------------------------------------
COL 19: 'अिधक'

ROW 5
--------------------------------------------------
COL 19: '200 kV &'

ROW 6
--------------------------------------------------
COL 19: 'Above'

ROW 7
--------------------------------------------------
COL 0: 'आं(cid:364) (cid:366)देश / Andhra

In [ ]:
# ==========================================================
# PHASE 10B
# WHEELING PARSER
# ==========================================================

import pdfplumber
import pandas as pd

# ----------------------------------------------------------
# VOLTAGE MAP
# ----------------------------------------------------------

WHEELING_COLS = {

    "Below 11 kV": 5,
    "11 kV": 8,
    "33 kV": 11,
    "66 kV": 14,
    "132 kV": 17,
    "200 kV & Above": 18
}

# ----------------------------------------------------------
# PARSER
# ----------------------------------------------------------

def parse_wheeling_matrix(df):

    records = []

    current_state = None

    for idx, row in df.iterrows():

        # Skip header section

        if idx <= 6:
            continue

        name = clean_text(
            extract_english(
                row.iloc[0]
            )
        )

        year = clean_text(
            row.iloc[2]
        )

        if not name:
            continue

        normalized = canonicalize_state(
            name
        )

        is_state = (
            normalized
            in canonical_states
        )

        # ----------------------------------
        # STATE ROW
        # ----------------------------------

        if is_state:

            current_state = normalized
            continue

        # ----------------------------------
        # UTILITY ROW
        # ----------------------------------

        if current_state is None:
            continue

        utility = name

        for voltage, col in WHEELING_COLS.items():

            value = clean_text(
                row.iloc[col]
            )

            if not value:
                continue

            records.append({

                "state":
                    current_state,

                "utility":
                    utility,

                "year":
                    year,

                "voltage_level":
                    voltage,

                "wheeling_charge":
                    value
            })

    return records


# ----------------------------------------------------------
# EXTRACTION
# ----------------------------------------------------------

def extract_wheeling():

    rows = []

    with pdfplumber.open(PDF_PATH) as pdf:

        for page_num in [59, 60]:

            page = pdf.pages[
                page_num - 1
            ]

            tables = page.extract_tables()

            if not tables:
                continue

            largest = max(
                tables,
                key=lambda t:
                    len(t)
                    *
                    max(
                        len(r)
                        for r in t
                        if r
                    )
            )

            rows.extend(
                largest
            )

    df = pd.DataFrame(rows)

    return parse_wheeling_matrix(df)


# ----------------------------------------------------------
# TEST
# ----------------------------------------------------------

wheeling = extract_wheeling()

print(
    "Records:",
    len(wheeling)
)

print("\nFIRST 30\n")

for r in wheeling[:30]:

    print(r)

Records: 93

FIRST 30

{'state': 'Andhra Pradesh', 'utility': 'APEPDCL', 'year': '2026-27', 'voltage_level': 'Below 11 kV', 'wheeling_charge': 'NA'}
{'state': 'Andhra Pradesh', 'utility': 'APEPDCL', 'year': '2026-27', 'voltage_level': '11 kV', 'wheeling_charge': '1.00'}
{'state': 'Andhra Pradesh', 'utility': 'APEPDCL', 'year': '2026-27', 'voltage_level': '33 kV', 'wheeling_charge': '0.47'}
{'state': 'Andhra Pradesh', 'utility': 'APEPDCL', 'year': '2026-27', 'voltage_level': '66 kV', 'wheeling_charge': '--'}
{'state': 'Andhra Pradesh', 'utility': 'APEPDCL', 'year': '2026-27', 'voltage_level': '132 kV', 'wheeling_charge': '0.28'}
{'state': 'Andhra Pradesh', 'utility': 'APEPDCL', 'year': '2026-27', 'voltage_level': '200 kV & Above', 'wheeling_charge': '0.28'}
{'state': 'Andhra Pradesh', 'utility': 'APSPDCL', 'year': '2026-27', 'voltage_level': 'Below 11 kV', 'wheeling_charge': 'NA'}
{'state': 'Andhra Pradesh', 'utility': 'APSPDCL', 'year': '2026-27', 'voltage_level': '11 kV', 'wheeling_ch

In [ ]:
# ==========================================================
# PHASE 11
# EXPORT ALL PARAMETERS
# ==========================================================

import pandas as pd

# ----------------------------------------------------------
# DATAFRAMES
# ----------------------------------------------------------

banking_df = pd.DataFrame(banking)

transmission_df = pd.DataFrame(transmission)

additional_df = pd.DataFrame(additional)

wheeling_df = pd.DataFrame(wheeling)

# ----------------------------------------------------------
# SUMMARY
# ----------------------------------------------------------

print("="*80)
print("DATASET SUMMARY")
print("="*80)

print(
    "Banking:",
    len(banking_df)
)

print(
    "Transmission:",
    len(transmission_df)
)

print(
    "Additional Surcharge:",
    len(additional_df)
)

print(
    "Wheeling:",
    len(wheeling_df)
)

# ----------------------------------------------------------
# EXPORT
# ----------------------------------------------------------

OUTPUT_FILE = (
    "Regulatory_Parameter_Warehouse.xlsx"
)

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    banking_df.to_excel(
        writer,
        sheet_name="Banking",
        index=False
    )

    transmission_df.to_excel(
        writer,
        sheet_name="Transmission",
        index=False
    )

    additional_df.to_excel(
        writer,
        sheet_name="Additional_Surcharge",
        index=False
    )

    wheeling_df.to_excel(
        writer,
        sheet_name="Wheeling",
        index=False
    )

print()
print("Saved:")
print(OUTPUT_FILE)

DATASET SUMMARY
Banking: 53
Transmission: 25
Additional Surcharge: 37
Wheeling: 93

Saved:
Regulatory_Parameter_Warehouse.xlsx


In [ ]:
# ==========================================================
# PHASE 12
# FORMAT WORKBOOK
# ==========================================================

from openpyxl import load_workbook
from openpyxl.styles import Font

FILE = (
    "Regulatory_Parameter_Warehouse.xlsx"
)

wb = load_workbook(FILE)

for ws in wb.worksheets:

    ws.freeze_panes = "A2"

    for cell in ws[1]:

        cell.font = Font(
            bold=True
        )

    for column in ws.columns:

        max_len = 0

        col_letter = (
            column[0]
            .column_letter
        )

        for cell in column:

            try:

                max_len = max(
                    max_len,
                    len(str(cell.value))
                )

            except:
                pass

        ws.column_dimensions[
            col_letter
        ].width = min(
            max_len + 2,
            60
        )

wb.save(FILE)

print("Workbook formatted.")
print(FILE)

Workbook formatted.
Regulatory_Parameter_Warehouse.xlsx
